# Detecting Query Performance Regression in SQL Server Index Tuning with a Rank-Average Ensemble of Cost-Sensitive Models
Pengukuran label (v2), evaluasi detektor pola Wu dan gabungan peringkat model, ablasi, SHAP, dan analisis galat.
Jalankan cell dari atas ke bawah, satu per satu. Jika ada error, hentikan dan periksa pesan error pada cell tersebut.

Prasyarat: SQL Server (database `DSB_SF10`) yang dapat dijangkau lewat Tailscale, akun Tailscale, dan folder `QPR_SF10_Research` di Google Drive. Kredensial tidak disimpan di notebook ini; IP dan kata sandi dimasukkan saat dijalankan.


## Setup

In [ ]:
# LANGKAH 1 - pasang alat (sekali per sesi Colab baru, +- 2-4 menit)
!curl -fsSL https://tailscale.com/install.sh | sh
!sudo apt-get install -y socat
!curl -sSL -O https://packages.microsoft.com/config/ubuntu/24.04/packages-microsoft-prod.deb
!sudo dpkg -i packages-microsoft-prod.deb
!sudo apt-get update -qq
!sudo ACCEPT_EULA=Y apt-get install -y msodbcsql18 unixodbc-dev
!pip install -q pyodbc pandas

In [ ]:
# LANGKAH 2 - hidupkan Tailscale
# Akan muncul link login. Klik link-nya, authorize, lalu tunggu cell ini selesai.
!sudo tailscaled --tun=userspace-networking --state=/tmp/tailscaled.state --socket=/tmp/tailscaled.sock >/tmp/tailscaled.log 2>&1 &
!sleep 4
!sudo tailscale --socket=/tmp/tailscaled.sock up --hostname=colab-qpr-remeasure

In [ ]:
# LANGKAH 3 - jembatan Colab -> SQL Server
WINDOWS_TS_IP = input("IP Tailscale komputer SQL Server (format 100.x.x.x): ").strip()
SQL_PORT = input("Port TCP SQL Server pada komputer tersebut (default 1433): ").strip() or "1433"
LOCAL_PORT = "11433"   # port lokal di Colab yang diteruskan ke SQL Server (bebas, asal tidak dipakai proses lain)

!sudo tailscale --socket=/tmp/tailscaled.sock status
!sudo tailscale --socket=/tmp/tailscaled.sock ping {WINDOWS_TS_IP}

!pkill -f "socat TCP-LISTEN:{LOCAL_PORT}" || true
!nohup socat TCP-LISTEN:{LOCAL_PORT},reuseaddr,fork EXEC:"tailscale --socket=/tmp/tailscaled.sock nc {WINDOWS_TS_IP} {SQL_PORT}" >/tmp/qpr-socat.log 2>&1 &
!sleep 3
!ss -ltnp | grep {LOCAL_PORT}

In [ ]:
# LANGKAH 4 - tes koneksi SQL Server
import pyodbc
from getpass import getpass

SQL_USER = input("SQL Server username: ").strip()
SQL_PASSWORD = getpass("SQL Server password: ")

CONN_STR = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER=127.0.0.1,{LOCAL_PORT};"
    "DATABASE=DSB_SF10;"
    f"UID={SQL_USER};"
    f"PWD={SQL_PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
    "Connection Timeout=30;"
)

conn = pyodbc.connect(CONN_STR, timeout=30)
cur = conn.cursor()
cur.execute("SELECT @@SERVERNAME, DB_NAME(), COUNT_BIG(*) FROM dbo.store_sales;")
row = cur.fetchone()
print("Server      :", row[0])
print("Database    :", row[1])
print("store_sales :", f"{row[2]:,}")
cur.close(); conn.close()
print("\nKONEKSI OK")

In [ ]:
# LANGKAH 5 - sambungkan Google Drive
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/QPR_SF10_Research")
INCREMENTAL_ROOT = DRIVE_ROOT / "incremental"

assert (INCREMENTAL_ROOT / "incremental_replay_final_valid.csv").exists(), \
    "incremental_replay_final_valid.csv tidak ditemukan di Drive"
print("Drive OK. Folder:", INCREMENTAL_ROOT)

In [ ]:
# LANGKAH 6 - upload zip DTA incremental (yang sama dengan eksperimen sebelumnya) -> bangun transitions_df
from google.colab import files
from pathlib import Path
import zipfile, shutil, json
import pandas as pd

uploaded = files.upload()
zip_name = next(n for n in uploaded if n.endswith(".zip"))
zip_path = Path("/content") / zip_name

EXTRACT_ROOT = Path("/content/dta_incremental")
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    for member in zf.infolist():
        clean_name = member.filename.replace("\\", "/").lstrip("/")
        if not clean_name:
            continue
        target = EXTRACT_ROOT / clean_name
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)

rows = []
for status_path in sorted(EXTRACT_ROOT.rglob("status.json")):
    try:
        status = json.loads(status_path.read_text(encoding="utf-8-sig"))
    except UnicodeDecodeError:
        status = json.loads(status_path.read_text(encoding="utf-16"))
    if status.get("index_recommended") is True:
        rows.append({"query_id": status.get("query_id"),
                     "iteration": int(status.get("iteration")),
                     "status_path": str(status_path)})

transitions_df = pd.DataFrame(rows).sort_values(["query_id", "iteration"]).reset_index(drop=True)
n_complete = len(list(EXTRACT_ROOT.rglob("_QUERY_COMPLETE.json")))
print("Query lengkap  :", n_complete, "(harus 74)")
print("Transisi index :", len(transitions_df), "(harus 325)")
assert n_complete == 74 and len(transitions_df) == 325

In [ ]:
# LANGKAH 7 - upload zip workload 74 query -> bangun query_map
from google.colab import files
from pathlib import Path
import zipfile, shutil

uploaded = files.upload()
zip_name = next(n for n in uploaded if n.endswith(".zip"))
WORKLOAD_ZIP = Path("/content") / zip_name
WORKLOAD_ROOT = Path("/content/workload_sf10")
if WORKLOAD_ROOT.exists():
    shutil.rmtree(WORKLOAD_ROOT)
WORKLOAD_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(WORKLOAD_ZIP, "r") as zf:
    for member in zf.infolist():
        clean_name = member.filename.replace("\\", "/").lstrip("/")
        if not clean_name:
            continue
        target = WORKLOAD_ROOT / clean_name
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)

query_files = [p for p in sorted(WORKLOAD_ROOT.rglob("*.sql")) if "tmp" not in str(p).lower()]
query_map = {p.stem: p for p in query_files}
print("File query:", len(query_files), "(harus 74)")
missing = sorted(set(transitions_df["query_id"]) - set(query_map))
print("Query tanpa file SQL:", len(missing))
assert len(query_files) == 74 and not missing

## Pengukuran ulang label (v2)
Default `LIMIT_QUERIES = None` (semua query). Isi angka (mis. 2) untuk uji coba; query yang sudah selesai dilewati otomatis.

In [ ]:
# ============================================================
# UKUR ULANG LABEL QPR (v2)
# Ide: tiap KONDISI index (C0, C1, C2, ...) diukur beberapa kali
#      (1x pemanasan dibuang, lalu 2-5x diukur, ambil MEDIAN).
#      Rasio transisi i = median(Ci) / median(Ci-1).
#      Jadi rantai before/after ikut konsisten.
# Hanya MEMBACA file label lama. Hasil ditulis ke folder baru "remeasure".
# Butuh sesi Colab yang sama dengan replay (CONN_STR, transitions_df,
# query_map sudah ada = Langkah 4, 6, dan 7 sudah dijalankan).
# ============================================================
import re
import time
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import pyodbc

# ---------------- PENGATURAN ----------------
LIMIT_QUERIES = None       # None = semua query yang belum selesai. Isi angka (mis. 2) untuk uji coba.
WARMUP_RUNS = 1            # lari pemanasan (dibuang)
THRESH = 1.5               # batas Regression, sama seperti sebelumnya
BORDER_LOW, BORDER_HIGH = 1.3, 1.7   # baris "dekat garis" -> diproses duluan
MIN_MS = 1.0               # SQL Server mengukur per 1 ms; hindari bagi nol

if "INCREMENTAL_ROOT" not in globals():      # kalau Langkah 5 belum dijalankan di sesi ini
    INCREMENTAL_ROOT = "/content/drive/MyDrive/QPR_SF10_Research/incremental"

_missing = [n for n in ["CONN_STR", "query_map", "transitions_df"] if n not in globals()]
assert not _missing, (
    f"Variabel ini belum ada di sesi: {_missing}. "
    "Jalankan ulang Langkah 4 (koneksi SQL), Langkah 6 (zip DTA), dan Langkah 7 (zip workload)."
)
assert Path(INCREMENTAL_ROOT).exists(), (
    "Folder Drive tidak ditemukan. Jalankan Langkah 5 (sambungkan Google Drive) dulu."
)

INCREMENTAL_ROOT = Path(INCREMENTAL_ROOT)
OUT_DIR = INCREMENTAL_ROOT / "remeasure"
OUT_DIR.mkdir(parents=True, exist_ok=True)
STATES_CSV = OUT_DIR / "remeasure_states.csv"
LABELS_CSV = OUT_DIR / "remeasure_labels_v2.csv"
VALID_CSV = INCREMENTAL_ROOT / "incremental_replay_final_valid.csv"
FEAT_CSV = INCREMENTAL_ROOT / "incremental_m0_m2b_features.csv"


# ---------------- ALAT BANTU SQL ----------------
def read_text_auto(path):
    path = Path(path)
    for enc in ["utf-8-sig", "utf-8", "utf-16", "utf-16-le", "cp1252"]:
        try:
            return path.read_text(encoding=enc)
        except UnicodeError:
            pass
    return path.read_text(encoding="utf-8", errors="ignore")


def split_batches(sql_text):
    parts = re.split(r"(?im)^\s*GO\s*(?:--.*)?$", sql_text)
    return [x.strip() for x in parts if x.strip()]


def clean_query_sql(sql_text):
    return "\n".join(
        ln for ln in sql_text.splitlines()
        if not re.match(r"^\s*GO\s*(?:--.*)?$", ln, re.I)
    ).strip()


import itertools
_conn_counter = itertools.count(1)


def connect(autocommit=True, timeout=30):
    """Koneksi baru dengan nama aplikasi unik, supaya tidak memakai ulang
    koneksi 'kotor' dari pool ODBC (penyebab error 'Connection is busy')."""
    cs = CONN_STR.rstrip().rstrip(";") + f";APP=qprremeasure{next(_conn_counter)};"
    return pyodbc.connect(cs, timeout=timeout, autocommit=autocommit)


def run_script(path):
    """Jalankan skrip DDL; tiap batch GO pakai koneksi sendiri."""
    for batch in split_batches(read_text_auto(path)):
        conn = connect()
        try:
            cur = conn.cursor()
            cur.execute(batch)
            while True:
                if cur.description is not None:
                    try:
                        cur.fetchall()
                    except pyodbc.Error:
                        pass
                try:
                    if not cur.nextset():
                        break
                except pyodbc.Error:
                    break
            cur.close()
        finally:
            conn.close()


def _grab(cur, bucket):
    try:
        for item in (getattr(cur, "messages", None) or []):
            text = str(item[-1]) if isinstance(item, tuple) else str(item)
            if text not in bucket:
                bucket.append(text)
    except Exception:
        pass


TIME_RE = re.compile(
    r"SQL Server Execution Times:\s*CPU time\s*=\s*(\d+)\s*ms,\s*elapsed time\s*=\s*(\d+)\s*ms",
    re.I | re.S,
)


def measure_once(query_sql):
    """Jalankan query 1x, kembalikan (elapsed_ms_server, cpu_ms, wall_ms, pakai_wall)."""
    conn = connect()
    msgs = []
    try:
        cur = conn.cursor()
        cur.execute("SET STATISTICS TIME ON;")
        t0 = time.perf_counter()
        cur.execute(query_sql)
        _grab(cur, msgs)
        while True:
            if cur.description is not None:
                while True:
                    rows = cur.fetchmany(1000)
                    _grab(cur, msgs)
                    if not rows:
                        break
            _grab(cur, msgs)
            try:
                more = cur.nextset()
                _grab(cur, msgs)
                if not more:
                    break
            except pyodbc.Error:
                break
        wall = (time.perf_counter() - t0) * 1000.0
        try:
            cur.execute("SET STATISTICS TIME OFF;")
        except Exception:
            pass
        cur.close()
    finally:
        conn.close()
    m = TIME_RE.findall("\n".join(msgs))
    if m:
        return float(sum(int(e) for _, e in m)), float(sum(int(c) for c, _ in m)), wall, False
    return wall, None, wall, True


def cleanup_query(path, tries=3):
    """Kembalikan query ke C0. Coba ulang kalau gagal; kalau tetap gagal, berhenti."""
    last = None
    for k in range(1, tries + 1):
        try:
            run_script(path)
            return
        except pyodbc.Error as e:
            last = e
            print(f"  [peringatan] pembersihan gagal (percobaan {k}/{tries}): {str(e)[:140]}")
            time.sleep(3)
    raise RuntimeError(f"Pembersihan index gagal {tries}x. Periksa database sebelum lanjut. Error terakhir: {last}")


def measure_state(query_sql):
    warm_ms = None
    for _ in range(WARMUP_RUNS):
        warm_ms = measure_once(query_sql)[0]
    base = warm_ms if warm_ms is not None else 0.0
    n = 5 if base < 5000 else (3 if base < 30000 else 2)
    runs, used_wall = [], 0
    for _ in range(n):
        e, _c, _w, uw = measure_once(query_sql)
        runs.append(e)
        used_wall += int(uw)
    arr = np.array(runs, dtype=float)
    return {
        "n_runs": n,
        "warmup_ms": warm_ms,
        "runs_ms": json.dumps([round(x, 3) for x in runs]),
        "median_ms": float(np.median(arr)),
        "min_ms": float(arr.min()),
        "max_ms": float(arr.max()),
        "cv": float(arr.std(ddof=1) / arr.mean()) if n > 1 and arr.mean() > 0 else 0.0,
        "used_wall_fallback": used_wall,
    }


# ---------------- DAFTAR KERJA ----------------
valid = pd.read_csv(VALID_CSV)
valid["query_id"] = valid["query_id"].astype(str)
trans = transitions_df.copy()
trans["query_id"] = trans["query_id"].astype(str)
trans["iteration"] = trans["iteration"].astype(int)

border = valid[(valid["label"] == 1) |
               ((valid["runtime_ratio"] >= BORDER_LOW) & (valid["runtime_ratio"] <= BORDER_HIGH))]
priority = list(dict.fromkeys(border["query_id"]))
others = [q for q in dict.fromkeys(valid["query_id"]) if q not in set(priority)]
order = priority + others
print(f"Query total: {len(order)} | diproses duluan (ada Regression/dekat garis): {len(priority)}")


def iters_of(qid):
    return list(trans[trans["query_id"] == qid].sort_values("iteration")["iteration"])


states_df = pd.read_csv(STATES_CSV) if STATES_CSV.exists() else pd.DataFrame(
    columns=["query_id", "state_iter", "n_runs", "warmup_ms", "runs_ms", "median_ms",
             "min_ms", "max_ms", "cv", "used_wall_fallback", "completed_at"])
states_df["query_id"] = states_df["query_id"].astype(str)

ERRORS_CSV = OUT_DIR / "remeasure_errors.csv"
errors_df = pd.read_csv(ERRORS_CSV) if ERRORS_CSV.exists() else pd.DataFrame(
    columns=["query_id", "failed_state", "reason", "note", "recorded_at"])
errors_df["query_id"] = errors_df["query_id"].astype(str)


def query_done(qid):
    if qid in set(errors_df["query_id"]):      # query error (mis. SQL 8134): dianggap selesai, sisanya dilewati
        return True
    have = set(states_df.loc[states_df["query_id"] == qid, "state_iter"].astype(int))
    return have == ({0} | set(iters_of(qid)))


todo = [q for q in order if not query_done(q)]
done_before = len(order) - len(todo)
if LIMIT_QUERIES is not None:
    todo = todo[:LIMIT_QUERIES]
print(f"Sudah selesai sebelumnya: {done_before} | akan diproses sekarang: {len(todo)}")

# ---------------- UKUR ----------------
for qid in todo:
    qt = trans[trans["query_id"] == qid].sort_values("iteration")
    first_status = Path(qt.iloc[0]["status_path"])
    cleanup_path = first_status.parent.parent / "cleanup_to_C0.sql"
    assert cleanup_path.exists(), f"cleanup_to_C0.sql tidak ada: {cleanup_path}"
    query_sql = clean_query_sql(read_text_auto(query_map[qid]))

    # buang sisa data parsial query ini (kalau pernah terputus)
    states_df = states_df[states_df["query_id"] != qid]
    print("\n" + "=" * 60)
    print("QUERY:", qid, f"| kondisi yang diukur: {len(qt) + 1}")
    print("=" * 60)

    def save_state(state_iter, res):
        global states_df
        row = {"query_id": qid, "state_iter": state_iter,
               "completed_at": datetime.now().isoformat(), **res}
        states_df = pd.concat([states_df, pd.DataFrame([row])], ignore_index=True)
        states_df.to_csv(STATES_CSV, index=False)
        print(f"  C{state_iter}: median={res['median_ms']:.0f} ms "
              f"(min {res['min_ms']:.0f}, max {res['max_ms']:.0f}, n={res['n_runs']}, cv={res['cv']:.2f})")

    current_state = 0
    try:
        cleanup_query(cleanup_path)                    # mulai dari C0
        try:
            save_state(0, measure_state(query_sql))
            for _, tr in qt.iterrows():
                it = int(tr["iteration"])
                current_state = it
                rec = Path(tr["status_path"]).parent / "recommendation.sql"
                run_script(rec)                        # C(i-1) -> C(i)
                save_state(it, measure_state(query_sql))
        except pyodbc.Error as e:
            msg = str(e)
            if "8134" in msg or "22012" in msg:        # divide by zero di dalam query itu sendiri
                print(f"  !! C{current_state}: query error SQL 8134 (divide by zero). "
                      f"Sisa kondisi query ini dilewati, tidak ada label yang dikarang.")
                errors_df = pd.concat([errors_df, pd.DataFrame([{
                    "query_id": qid, "failed_state": current_state,
                    "reason": "SQL_SERVER_ERROR_8134_DIVIDE_BY_ZERO",
                    "note": "Kondisi ini gagal dieksekusi; kondisi berikutnya dilewati",
                    "recorded_at": datetime.now().isoformat()}])], ignore_index=True)
                errors_df.to_csv(ERRORS_CSV, index=False)
            else:
                raise
    finally:
        cleanup_query(cleanup_path)                    # selalu balik ke C0

# ---------------- VERIFIKASI DB BERSIH ----------------
conn = connect(autocommit=False)
cur = conn.cursor()
cur.execute("""SELECT (SELECT COUNT(*) FROM sys.indexes WHERE name LIKE '_dta_%'),
                      (SELECT COUNT(*) FROM sys.stats   WHERE name LIKE '_dta_%')""")
n_idx, n_stat = cur.fetchone()
cur.close(); conn.close()
print(f"\nSisa objek _dta_: index={n_idx}, stats={n_stat}", "-> DB BERSIH" if (n_idx == 0 and n_stat == 0) else "-> PERIKSA!")

# ---------------- HITUNG LABEL v2 ----------------
rows = []
for qid in order:
    if not query_done(qid):
        continue
    seq = [0] + iters_of(qid)
    st = states_df[states_df["query_id"] == qid].set_index(states_df[states_df["query_id"] == qid]["state_iter"].astype(int))
    for k in range(1, len(seq)):
        it, prev = seq[k], seq[k - 1]
        v = valid[(valid["query_id"] == qid) & (valid["iteration"] == it)]
        if v.empty:
            continue
        v = v.iloc[0]
        if prev not in st.index or it not in st.index:     # kondisi tidak diukur (query error)
            continue
        b, a = st.loc[prev], st.loc[it]
        bm, am = max(float(b["median_ms"]), MIN_MS), max(float(a["median_ms"]), MIN_MS)
        ratio = am / bm
        lo = max(float(a["min_ms"]), MIN_MS) / max(float(b["max_ms"]), MIN_MS)
        hi = max(float(a["max_ms"]), MIN_MS) / max(float(b["min_ms"]), MIN_MS)
        certainty = "REG_PASTI" if lo >= THRESH else ("NORMAL_PASTI" if hi < THRESH else "TIDAK_PASTI")
        rows.append({
            "sample_id": v["sample_id"], "query_id": qid, "iteration": it,
            "before_v1_ms": v["before_elapsed_ms"], "after_v1_ms": v["after_elapsed_ms"],
            "ratio_v1": v["runtime_ratio"], "label_v1": int(v["label"]),
            "before_v2_ms": bm, "after_v2_ms": am, "ratio_v2": ratio,
            "label_v2": int(ratio >= THRESH), "ratio_terbaik": lo, "ratio_terburuk": hi,
            "kepastian_v2": certainty, "delta_v2_s": (am - bm) / 1000.0,
            "cv_before": float(b["cv"]), "cv_after": float(a["cv"]),
        })
lab = pd.DataFrame(rows)
if len(lab):
    lab["berubah"] = (lab["label_v1"] != lab["label_v2"]).astype(int)
    lab.to_csv(LABELS_CSV, index=False)

# ---------------- RINGKASAN ----------------
print("\n" + "=" * 60)
print("RINGKASAN UKUR ULANG")
print("=" * 60)
if not len(lab):
    print("Belum ada query yang selesai.")
else:
    n_left = len([q for q in order if not query_done(q)])
    print(f"Transisi yang sudah diukur ulang : {len(lab)} dari {len(valid)}"
          + (f"  (BARU SEBAGIAN: {n_left} query belum selesai)" if n_left else ""))
    if len(errors_df):
        print(f"Query dilewati karena error SQL  : {', '.join(errors_df['query_id'])} "
              f"(sisa transisinya tidak punya label v2)")
    print(f"Regression v1 -> v2              : {int(lab['label_v1'].sum())} -> {int(lab['label_v2'].sum())}")
    ct = pd.crosstab(lab["label_v1"], lab["label_v2"]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    print("Tabel perubahan (baris=v1, kolom=v2):")
    print(ct.to_string())
    print(f"Label berubah                    : {int(lab['berubah'].sum())} dari {len(lab)}")
    print("Kepastian label v2:")
    print(lab["kepastian_v2"].value_counts().to_string())
    r2 = lab[lab["label_v2"] == 1]
    if len(r2):
        print(f"Regression v2 dengan selisih >= 0,1 dtk: {(r2['delta_v2_s'] >= 0.1).sum()} dari {len(r2)}")
        print(f"Regression v2 dengan selisih >= 1 dtk  : {(r2['delta_v2_s'] >= 1.0).sum()} dari {len(r2)}")
    print(f"Median CV pengukuran (noise)     : {lab[['cv_before','cv_after']].stack().median():.3f}")
    print(f"Baris dengan before < 10 ms (resolusi kasar): {(lab['before_v2_ms'] < 10).sum()}")

    if FEAT_CSV.exists():
        f = pd.read_csv(FEAT_CSV, usecols=lambda c: c in ("sample_id", "m0_prediction"))
        if "m0_prediction" in f.columns:
            m = lab.merge(f, on="sample_id", how="inner")

            def prf(y, p):
                tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
                fn = int(((y == 1) & (p == 0)).sum())
                pr = tp / (tp + fp) if tp + fp else 0.0
                rc = tp / (tp + fn) if tp + fn else 0.0
                f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
                return tp, fp, fn, pr, rc, f1
            print("\nM0 (Wu) pada label lama vs label baru (pada transisi yang sama):")
            for name in ["label_v1", "label_v2"]:
                tp, fp, fn, pr, rc, f1 = prf(m[name].to_numpy(), m["m0_prediction"].to_numpy())
                print(f"  {name}: TP={tp} FP={fp} FN={fn} | P={pr:.3f} R={rc:.3f} F1={f1:.3f}")
    print("\nFile hasil:")
    print("  ", STATES_CSV)
    print("  ", LABELS_CSV)
    if LIMIT_QUERIES is not None:
        print("\nMode uji coba aktif (LIMIT_QUERIES terisi). Jika tidak ada error, ubah LIMIT_QUERIES = None lalu jalankan ulang")
        print("(yang sudah selesai dilewati otomatis, jadi aman diulang/dilanjutkan).")

## Evaluasi utama: Detektor Pola Wu vs model kardinalitas lokal pada label v1 dan v2


In [ ]:
# ============================================================
# EVALUASI M0 vs M2b pada label v1 dan v2  (protokol dikunci)
# ------------------------------------------------------------
# DIKUNCI SEBELUM MELIHAT HASIL:
#  - Metrik utama      : F1 kelas Regression (M2b vs M0, baris yang sama)
#  - Metrik pendukung  : Precision, Recall, MCC, PR-AUC (hanya M2b)
#  - Model M2b         : sama persis dengan evaluasi sebelumnya (XGBoost, fitur M2b,
#                        5-fold grouped by template, threshold dipilih dari data latih saja)
#  - Label             : v1 (lama) dan v2 (ukur ulang) dilaporkan dua-duanya
#  - Tambahan sensitivitas: v2_tegas = v2 tanpa baris TIDAK_PASTI
#  - Tidak ada tuning apa pun setelah melihat hasil
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (average_precision_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef)

ROOT = Path("/content/drive/MyDrive/QPR_SF10_Research/incremental")
FEATURE_CSV = ROOT / "incremental_m0_m2b_features.csv"
LABEL_V2_CSV = ROOT / "remeasure" / "remeasure_labels_v2.csv"
OUT_CSV = ROOT / "remeasure" / "eval_m0_vs_m2b.csv"

CANON_SEED = 42                 # seed utama (sama dengan evaluasi sebelumnya)
EXTRA_SEEDS = [43, 44, 45, 46]  # untuk cek kestabilan terhadap pembagian fold
N_BOOT = 2000
BOOT_SEED = 2026

# ---------- load & gabung label ----------
feat = pd.read_csv(FEATURE_CSV)
lab2 = pd.read_csv(LABEL_V2_CSV)
assert "sample_id" in feat.columns, "Kolom sample_id tidak ada di file fitur"
need2 = ["sample_id", "label_v1", "label_v2", "kepastian_v2"]
df = feat.merge(lab2[need2], on="sample_id", how="inner")
print(f"Fitur: {len(feat)} baris | gabung dengan label v2: {len(df)} baris")
assert len(df) > 0
if len(df) != len(feat):
    print(f"  PERINGATAN: {len(feat) - len(df)} baris tidak punya label v2 (dibuang dari semua evaluasi)")
df["label_v1_file"] = df["label"].astype(int)
assert (df["label_v1_file"] == df["label_v1"].astype(int)).all(), "label v1 di file fitur beda dengan label_v1 di file v2"

# ---------- fitur M2b (aturan sama dengan evaluasi sebelumnya) ----------
forbidden = {"before_elapsed_ms", "after_elapsed_ms", "runtime_ratio", "label",
             "label_v1", "label_v2", "label_v1_file", "kepastian_v2"}
feature_cols = []
for col in df.columns:
    if col in forbidden or col.startswith("m0_"):
        continue
    if (col.startswith("before_") or col.startswith("candidate_") or col.startswith("delta_")
            or col.startswith("ce_") or col.startswith("loc_") or col == "estimated_cost_ratio"):
        if "elapsed_ms" in col:
            continue
        feature_cols.append(col)
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
X_all = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
groups_all = df["template_id"].astype(str).to_numpy()
m0_all = df["m0_prediction"].astype(int).to_numpy()
print("Fitur M2b:", len(feature_cols), "| template:", len(np.unique(groups_all)))

def build_model(y_train):
    pos = int((y_train == 1).sum()); neg = int((y_train == 0).sum())
    return XGBClassifier(objective="binary:logistic", n_estimators=200, learning_rate=0.05,
                         max_depth=3, min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                         reg_lambda=2.0, reg_alpha=0.0, scale_pos_weight=neg / max(pos, 1),
                         eval_metric="logloss", tree_method="hist", n_jobs=2,
                         random_state=42, verbosity=0)

def select_threshold(Xtr, ytr, gtr):
    n_pos_groups = len(set(gtr[ytr == 1]))
    k = min(3, max(n_pos_groups, 2))
    inner = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=123)
    prob = np.zeros(len(ytr))
    for tr, va in inner.split(Xtr, ytr, gtr):
        if len(set(ytr[tr])) < 2:
            continue
        m = build_model(ytr[tr]); m.fit(Xtr.iloc[tr], ytr[tr])
        prob[va] = m.predict_proba(Xtr.iloc[va])[:, 1]
    best, best_t = None, 0.5
    for t in np.linspace(0.05, 0.90, 86):
        p = (prob >= t).astype(int)
        s = (f1_score(ytr, p, zero_division=0), matthews_corrcoef(ytr, p),
             recall_score(ytr, p, zero_division=0), precision_score(ytr, p, zero_division=0))
        if best is None or s > best:
            best, best_t = s, float(t)
    return best_t

def run_m2b(X, y, groups, seed):
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    prob = np.zeros(len(y)); pred = np.zeros(len(y), dtype=int)
    for tr, te in outer.split(X, y, groups):
        Xtr = X.iloc[tr].reset_index(drop=True)
        thr = select_threshold(Xtr, y[tr], groups[tr])
        m = build_model(y[tr]); m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        prob[te] = p; pred[te] = (p >= thr).astype(int)
    return prob, pred

def f1_fast(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

def metrics(y, p, prob=None):
    d = {"Precision": precision_score(y, p, zero_division=0), "Recall": recall_score(y, p, zero_division=0),
         "F1": f1_score(y, p, zero_division=0), "MCC": matthews_corrcoef(y, p),
         "TP": int(((y == 1) & (p == 1)).sum()), "FP": int(((y == 0) & (p == 1)).sum()),
         "FN": int(((y == 1) & (p == 0)).sum())}
    if prob is not None and y.sum() > 0:
        d["PR_AUC"] = average_precision_score(y, prob)
    return d

def cluster_bootstrap(y, p0, p2, groups, n_boot, seed):
    rng = np.random.default_rng(seed)
    ug = np.unique(groups); idx = {g: np.where(groups == g)[0] for g in ug}
    f0, f2 = [], []
    for _ in range(n_boot):
        pick = rng.choice(ug, size=len(ug), replace=True)
        ii = np.concatenate([idx[g] for g in pick])
        f0.append(f1_fast(y[ii], p0[ii])); f2.append(f1_fast(y[ii], p2[ii]))
    f0 = np.array(f0); f2 = np.array(f2); d = f2 - f0
    ci = lambda a: (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
    return ci(f0), ci(f2), ci(d), float((d <= 0).mean())

def mcnemar_exact(y, p0, p2, mask):
    """McNemar exact pada subset baris (mask). Pada baris Regression: siapa yang lebih banyak menangkap.
    Pada baris Normal: siapa yang lebih banyak salah alarm."""
    a = int(((p2 == 1) & (p0 == 0) & mask).sum())   # hanya M2b yang bilang 'Regression'
    b = int(((p2 == 0) & (p0 == 1) & mask).sum())   # hanya M0 yang bilang 'Regression'
    p = binomtest(a, a + b, 0.5).pvalue if (a + b) > 0 else 1.0
    return a, b, p

label_sets = {
    "v1": ("label_v1", None),
    "v2": ("label_v2", None),
    "v2_tegas (tanpa TIDAK_PASTI)": ("label_v2", df["kepastian_v2"] != "TIDAK_PASTI"),
}

rows = []
for name, (col, mask) in label_sets.items():
    d = df if mask is None else df[mask.to_numpy()]
    X = X_all.loc[d.index].reset_index(drop=True)
    y = d[col].astype(int).to_numpy(); g = groups_all[d.index]; p0 = m0_all[d.index]
    print("\n" + "=" * 70)
    print(f"LABEL: {name} | baris={len(d)} | Regression={int(y.sum())} | template={len(np.unique(g))}")
    print("=" * 70)
    if y.sum() < 5:
        print("  Regression terlalu sedikit untuk dievaluasi."); continue

    prob, p2 = run_m2b(X, y, g, CANON_SEED)
    m0 = metrics(y, p0); m2 = metrics(y, p2, prob)
    ci0, ci2, cid, p_boot = cluster_bootstrap(y, p0, p2, g, N_BOOT, BOOT_SEED)
    ra, rb, rp = mcnemar_exact(y, p0, p2, y == 1)
    na, nb, np_ = mcnemar_exact(y, p0, p2, y == 0)

    f1s = []
    for s in EXTRA_SEEDS:
        _, ps = run_m2b(X, y, g, s); f1s.append(f1_score(y, ps, zero_division=0))
    f1_all = [m2["F1"]] + f1s

    print(f"{'':10s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s} {'MCC':>6s}  TP  FP  FN   F1 95% CI (bootstrap per template)")
    print(f"{'M0 (Wu)':10s} {m0['Precision']:6.3f} {m0['Recall']:6.3f} {m0['F1']:6.3f} {m0['MCC']:6.3f} {m0['TP']:3d} {m0['FP']:3d} {m0['FN']:3d}   [{ci0[0]:.3f}, {ci0[1]:.3f}]")
    print(f"{'M2b':10s} {m2['Precision']:6.3f} {m2['Recall']:6.3f} {m2['F1']:6.3f} {m2['MCC']:6.3f} {m2['TP']:3d} {m2['FP']:3d} {m2['FN']:3d}   [{ci2[0]:.3f}, {ci2[1]:.3f}]   PR-AUC={m2['PR_AUC']:.3f}")
    print(f"Selisih F1 (M2b - M0) = {m2['F1'] - m0['F1']:+.3f} | 95% CI [{cid[0]:+.3f}, {cid[1]:+.3f}] | bootstrap P(selisih<=0)={p_boot:.3f}")
    print(f"McNemar pada baris Regression (yang ditangkap): hanya M2b={ra}, hanya M0={rb}, p={rp:.4f}")
    print(f"McNemar pada baris Normal (salah alarm)      : hanya M2b={na}, hanya M0={nb}, p={np_:.4f}")
    print(f"Stabilitas F1 M2b di 5 seed fold: min={min(f1_all):.3f} median={np.median(f1_all):.3f} max={max(f1_all):.3f}")
    rows.append({"label_set": name, "n": len(d), "n_regression": int(y.sum()),
                 "M0_F1": m0["F1"], "M0_P": m0["Precision"], "M0_R": m0["Recall"],
                 "M2b_F1": m2["F1"], "M2b_P": m2["Precision"], "M2b_R": m2["Recall"], "M2b_PR_AUC": m2["PR_AUC"],
                 "dF1": m2["F1"] - m0["F1"], "dF1_CI_low": cid[0], "dF1_CI_high": cid[1],
                 "boot_P_le0": p_boot, "reg_only_M2b": ra, "reg_only_M0": rb, "reg_mcnemar_p": rp, "norm_only_M2b": na, "norm_only_M0": nb, "norm_mcnemar_p": np_,
                 "M2b_F1_seed_min": min(f1_all), "M2b_F1_seed_median": float(np.median(f1_all)), "M2b_F1_seed_max": max(f1_all)})

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print("\nTersimpan:", OUT_CSV)
print("\nCARA BACA: kalau CI selisih F1 melewati 0, atau p McNemar >= 0,05, jangan klaim M2b 'unggul'.")
print("Cukup tulis: ada indikasi, belum terbukti (data kecil, Regression sedikit).")

## Metrik pelengkap (ROC-AUC, Balanced Accuracy, confusion matrix)


In [ ]:
# ============================================================
# METRIK PELENGKAP (ROC-AUC, Balanced Accuracy, Confusion Matrix)
# ------------------------------------------------------------
# Kenapa: metrik ini direncanakan pada desain evaluasi tetapi belum dilaporkan.
# ATURAN:
#  - Model, fitur, threshold SAMA PERSIS dengan evaluasi terkunci (latih di data lama, seed 42).
#  - Tidak ada tuning dan tidak ada pemilihan ulang. Ini hanya menghitung metrik tambahan
#    dari prediksi yang sama. Angka pada evaluasi terkunci tidak boleh berubah; ada cek konsistensi di bawah.
#  - M0 hanya punya keputusan 0/1 (tanpa skor). Jadi ROC-AUC M0 dihitung dari keputusan 0/1,
#    dan nilainya sama dengan Balanced Accuracy. Ini satu titik operasi, bukan kurva.
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (average_precision_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, balanced_accuracy_score, confusion_matrix)

DRIVE_ROOT = Path("/content/drive/MyDrive/QPR_SF10_Research")
OLD_ROOT = DRIVE_ROOT / "incremental"
NEW_ROOT = INCREMENTAL_ROOT if "INCREMENTAL_ROOT" in globals() and "fresh" in str(INCREMENTAL_ROOT) \
    else DRIVE_ROOT / "fresh" / "incremental_fresh"
OLD_FEAT = OLD_ROOT / "incremental_m0_m2b_features.csv"
OLD_V2 = OLD_ROOT / "remeasure" / "remeasure_labels_v2.csv"
NEW_FEAT = NEW_ROOT / "incremental_m0_m2b_features.csv"
NEW_V2 = NEW_ROOT / "remeasure" / "remeasure_labels_v2.csv"
OUT_CSV = NEW_ROOT / "remeasure" / "eval_fresh_extra_metrics.csv"
for p in [OLD_FEAT, OLD_V2, NEW_FEAT, NEW_V2]:
    assert p.exists(), f"File tidak ditemukan: {p}"

old = pd.read_csv(OLD_FEAT).merge(pd.read_csv(OLD_V2)[["sample_id", "label_v1", "label_v2"]], on="sample_id", how="inner")
new = pd.read_csv(NEW_FEAT)
new["label_v1"] = new["label"].astype(int)
new = new.merge(pd.read_csv(NEW_V2)[["sample_id", "label_v2", "kepastian_v2"]], on="sample_id", how="left")

forbidden = {"before_elapsed_ms", "after_elapsed_ms", "runtime_ratio", "label",
             "label_v1", "label_v2", "kepastian_v2", "label_v1_file"}
feature_cols = []
for col in old.columns:
    if col in forbidden or col.startswith("m0_"):
        continue
    if (col.startswith("before_") or col.startswith("candidate_") or col.startswith("delta_")
            or col.startswith("ce_") or col.startswith("loc_") or col == "estimated_cost_ratio"):
        if "elapsed_ms" in col:
            continue
        feature_cols.append(col)
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(old[c])]
assert all(c in new.columns for c in feature_cols), "Ada fitur yang tidak ada di data baru"

def prep(d):
    return d[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

X_old = prep(old).reset_index(drop=True)
g_old = old["template_id"].astype(str).to_numpy()
X_new = prep(new).reset_index(drop=True)
m0_new = new["m0_prediction"].astype(int).to_numpy()

def build_model(y_train, seed):
    pos = int((y_train == 1).sum()); neg = int((y_train == 0).sum())
    return XGBClassifier(objective="binary:logistic", n_estimators=200, learning_rate=0.05,
                         max_depth=3, min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
                         reg_lambda=2.0, reg_alpha=0.0, scale_pos_weight=neg / max(pos, 1),
                         eval_metric="logloss", tree_method="hist", n_jobs=2,
                         random_state=seed, verbosity=0)

def select_threshold(Xtr, ytr, gtr):
    n_pos_groups = len(set(gtr[ytr == 1]))
    k = min(3, max(n_pos_groups, 2))
    inner = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=123)
    prob = np.zeros(len(ytr))
    for tr, va in inner.split(Xtr, ytr, gtr):
        if len(set(ytr[tr])) < 2:
            continue
        m = build_model(ytr[tr], 42); m.fit(Xtr.iloc[tr], ytr[tr])
        prob[va] = m.predict_proba(Xtr.iloc[va])[:, 1]
    best, best_t = None, 0.5
    for t in np.linspace(0.05, 0.90, 86):
        p = (prob >= t).astype(int)
        s = (f1_score(ytr, p, zero_division=0), matthews_corrcoef(ytr, p),
             recall_score(ytr, p, zero_division=0), precision_score(ytr, p, zero_division=0))
        if best is None or s > best:
            best, best_t = s, float(t)
    return best_t

def full_metrics(y, p, score=None):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    d = {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
         "Precision": precision_score(y, p, zero_division=0), "Recall": recall_score(y, p, zero_division=0),
         "Specificity": spec, "BalancedAcc": balanced_accuracy_score(y, p),
         "F1": f1_score(y, p, zero_division=0), "MCC": matthews_corrcoef(y, p)}
    d["ROC_AUC"] = roc_auc_score(y, score if score is not None else p)
    d["PR_AUC"] = average_precision_score(y, score) if score is not None else np.nan
    return d

def show_cm(name, m):
    print(f"  {name}")
    print(f"                  prediksi Normal   prediksi Regression")
    print(f"   aktual Normal   {m['TN']:10d}        {m['FP']:10d}")
    print(f"   aktual Regress. {m['FN']:10d}        {m['TP']:10d}")

sets = [("v2 (UTAMA)", "label_v2", "label_v2", None),
        ("v1 (protokol lama)", "label_v1", "label_v1", None),
        ("v2_tegas (tanpa TIDAK_PASTI)", "label_v2", "label_v2", new["kepastian_v2"] != "TIDAK_PASTI")]
rows = []
for name, col, train_col, extra in sets:
    y_tr = old[train_col].astype(int).to_numpy()
    valid = new[col].notna().to_numpy().copy()
    if extra is not None:
        valid = valid & extra.to_numpy()
    yt = new.loc[valid, col].astype(int).to_numpy()
    Xt = X_new.loc[valid].reset_index(drop=True); p0 = m0_new[valid]
    thr = select_threshold(X_old, y_tr, g_old)
    m = build_model(y_tr, 42); m.fit(X_old, y_tr)
    prob = m.predict_proba(Xt)[:, 1]; p2 = (prob >= thr).astype(int)
    r0 = full_metrics(yt, p0); r2 = full_metrics(yt, p2, prob)
    print("\n" + "=" * 72)
    print(f"LABEL UJI: {name} | uji: {len(yt)} baris | Regression={int(yt.sum())} | threshold={thr:.2f}")
    print("=" * 72)
    show_cm("M0 (Wu)", r0); show_cm("M2b", r2)
    keys = ["Precision", "Recall", "Specificity", "BalancedAcc", "F1", "MCC", "ROC_AUC", "PR_AUC"]
    print(f"\n  {'':8s}" + "".join(f"{k:>12s}" for k in keys))
    for lab, r in (("M0", r0), ("M2b", r2)):
        print(f"  {lab:8s}" + "".join(f"{r[k]:12.3f}" for k in keys))
    print("  Catatan: ROC-AUC M0 dihitung dari keputusan 0/1, jadi sama dengan Balanced Accuracy.")
    for lab, r in (("M0", r0), ("M2b", r2)):
        rows.append({"label_uji": name, "metode": lab, "n": len(yt), "n_regression": int(yt.sum()), "threshold": thr if lab == "M2b" else np.nan, **r})
    if name.startswith("v2 (UTAMA)"):
        ok = (r0["TP"], r0["FP"], r2["TP"], r2["FP"]) == (9, 32, 21, 89)
        print("\n  CEK KONSISTENSI dengan evaluasi sebelumnya (M0 TP=9 FP=32; M2b TP=21 FP=89):", "SAMA" if ok else
              "BEDA! Hentikan dan periksa pengaturan sebelum memakai angka apa pun.")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print("\nTersimpan:", OUT_CSV)

## Grafik hasil evaluasi


In [ ]:
# ============================================================
# GRAFIK HASIL (dibaca langsung dari file CSV hasil evaluasi terkunci, anggaran sama, metrik pelengkap, dan evaluasi data lama)
# ------------------------------------------------------------
# Tidak menghitung ulang apa pun dan tidak menyentuh model. Hanya menggambar angka yang sudah tersimpan.
# Gambar disimpan sebagai PNG di folder  fresh/incremental_fresh/remeasure/grafik/
# Syarat: ketiga evaluasi tersebut sudah pernah dijalankan (file CSV-nya ada di Drive).
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

DRIVE_ROOT = Path("/content/drive/MyDrive/QPR_SF10_Research")
NEW_ROOT = INCREMENTAL_ROOT if "INCREMENTAL_ROOT" in globals() and "fresh" in str(INCREMENTAL_ROOT) \
    else DRIVE_ROOT / "fresh" / "incremental_fresh"
REM = NEW_ROOT / "remeasure"
FRESH_CSV = REM / "eval_fresh_test.csv"               # evaluasi terkunci
BUDGET_CSV = REM / "eval_fresh_budget_matched.csv"    # anggaran sama
EXTRA_CSV = REM / "eval_fresh_extra_metrics.csv"      # metrik pelengkap
OLD_CSV = DRIVE_ROOT / "incremental" / "remeasure" / "eval_m0_vs_m2b.csv"   # evaluasi data lama (opsional)
OUT = REM / "grafik"; OUT.mkdir(parents=True, exist_ok=True)
FIG1_TAMPILKAN_SALAH_ALARM = False   # True = grafik 1 ditampilkan dua panel (tertangkap + salah alarm)

for p in [FRESH_CSV, BUDGET_CSV, EXTRA_CSV]:
    assert p.exists(), f"File belum ada: {p}. Jalankan dulu cell yang menghasilkannya."
fresh = pd.read_csv(FRESH_CSV); budget = pd.read_csv(BUDGET_CSV); extra = pd.read_csv(EXTRA_CSV)
old = pd.read_csv(OLD_CSV) if OLD_CSV.exists() else None

def main_row(df, col="label_uji"):
    r = df[df[col].astype(str).str.contains("UTAMA")]
    assert len(r), "Baris label v2 (UTAMA) tidak ditemukan"
    return r.iloc[0]
def c(x, nd=3):  # angka dengan koma desimal
    return f"{x:.{nd}f}".replace(".", ",")
def sg(x, nd=3):
    return f"{x:+.{nd}f}".replace(".", ",")

SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2dd"
C_M0, C_M2B, C_RND = "#2a78d6", "#eb6834", "#b9b8b2"      # M0 = biru, M2b = oranye
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "font.family": "DejaVu Sans", "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2,
    "ytick.color": INK2, "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110})

def clean(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, lw=0.8); ax.set_axisbelow(True); ax.tick_params(length=0)
    ax.spines["left" if axis == "y" else "bottom"].set_visible(False)

def bars(ax, x, h, colors, w=0.55, nd=None):
    b = ax.bar(x, h, width=w, color=colors, zorder=3)
    for r in b:
        v = r.get_height(); t = c(v, 1) if (nd == 1 and v != int(v)) else str(int(round(v)))
        ax.text(r.get_x() + r.get_width()/2, v, t, ha="center", va="bottom", fontsize=13, fontweight="bold", zorder=4)

def finish(fig, name, title, foot, top=0.93, bottom=0.04):
    fig.suptitle(title, fontsize=13.5, fontweight="bold", x=0.02, ha="left")
    fig.text(0.02, 0.005, foot, fontsize=9, color=INK2)
    fig.tight_layout(rect=(0, bottom, 1, top)); fig.savefig(OUT / name, dpi=200); plt.show(); plt.close(fig)

# data utama (label v2, uji segar)
fr = main_row(fresh); bu = main_row(budget)
e0 = extra[(extra["label_uji"].str.contains("UTAMA")) & (extra["metode"] == "M0")].iloc[0]
e2 = extra[(extra["label_uji"].str.contains("UTAMA")) & (extra["metode"] == "M2b")].iloc[0]
n_reg = int(e0["TP"] + e0["FN"]); n_norm = int(e0["TN"] + e0["FP"])

# ---------- 1. tertangkap (dan salah alarm, kalau diaktifkan) ----------
dua = FIG1_TAMPILKAN_SALAH_ALARM
fig, ax = plt.subplots(1, 2 if dua else 1, figsize=(9.5, 4.6) if dua else (6.2, 4.6))
ax = list(ax) if dua else [ax]
bars(ax[0], [0, 1], [e0["TP"], e2["TP"]], [C_M0, C_M2B]); ax[0].set_ylim(0, max(e0["TP"], e2["TP"]) * 1.6)
ax[0].set_title(f"Regression yang tertangkap\n(dari {n_reg} yang ada)", fontsize=12, loc="left"); clean(ax[0])
ax[0].text(-0.3, ax[0].get_ylim()[1] * 0.96, "Makin tinggi makin bagus", fontsize=9.5, color=INK2)
if dua:
    bars(ax[1], [0, 1], [e0["FP"], e2["FP"]], [C_M0, C_M2B]); ax[1].set_ylim(0, max(e0["FP"], e2["FP"]) * 1.2)
    ax[1].set_title(f"Salah alarm\n(query Normal yang ikut ditandai, dari {n_norm})", fontsize=12, loc="left"); clean(ax[1])
    ax[1].text(-0.3, ax[1].get_ylim()[1] * 0.96, "Makin rendah makin bagus", fontsize=9.5, color=INK2)
for a_ in ax: a_.set_xticks([0, 1], ["Metode Wu\n(M0)", "M2b\n(model saya)"], fontsize=11)
if dua:
    finish(fig, "fig1_tertangkap_salah_alarm.png", "M2b menangkap lebih banyak Regression, tapi lebih sering salah alarm",
           f"Data uji baru, label v2 (utama). Uji McNemar pada Regression yang tertangkap: p = {c(fr['reg_p'], 4)}.")

# ---------- 2. anggaran sama ----------
k = int(bu["k_M0"]); base = bu["n_regression"] / bu["n"]
fig, ax = plt.subplots(figsize=(8, 4.6))
bars(ax, [0, 1, 2], [k * base, bu["M0_TP"], bu["M2b_TP"]], [C_RND, C_M0, C_M2B], w=0.5, nd=1)
ax.set_ylim(0, bu["M2b_TP"] * 1.25); ax.set_ylabel(f"Regression tertangkap (dari {int(bu['n_regression'])})")
ax.set_xticks([0, 1, 2], [f"Tebak acak\n({k} query)", f"Metode Wu (M0)\n({k} query)", f"M2b\n({k} query teratas)"], fontsize=11)
clean(ax)
finish(fig, "fig2_anggaran_sama.png", f"Kalau keduanya menandai jumlah yang sama ({k} query)",
       f"EKSPLORATIF. Selisih F1 {sg(bu['dF1'])}; kisaran {sg(bu['dF1_CI_low'])} sampai {sg(bu['dF1_CI_high'])}. Label v2.")

# ---------- 3. selisih F1 + kisaran kepercayaan ----------
rows = []
for _, r in fresh.iterrows():
    rows.append(("Data baru, " + str(r["label_uji"]).replace(" (UTAMA)", "").split(" (")[0] + (" (UTAMA)" if "UTAMA" in str(r["label_uji"]) else ""),
                 r["dF1"], r["dF1_CI_low"], r["dF1_CI_high"], "UTAMA" in str(r["label_uji"])))
if old is not None:
    for _, r in old.iterrows():
        rows.append(("Data lama, " + str(r["label_set"]).split(" (")[0], r["dF1"], r["dF1_CI_low"], r["dF1_CI_high"], False))
rows.append((f"Jumlah tanda sama (eksploratif)", bu["dF1"], bu["dF1_CI_low"], bu["dF1_CI_high"], False))
lo = min(r[2] for r in rows) - 0.06; hi = max(r[3] for r in rows) + 0.04; col = hi + 0.1
fig, ax = plt.subplots(figsize=(9.5, 5)); y = np.arange(len(rows))[::-1]
ax.axvspan(lo, 0, color="#f1efe9", zorder=0); ax.axvline(0, color=INK2, lw=1.4, zorder=2)
for yi, (lab, d, l, h, main) in zip(y, rows):
    ax.plot([l, h], [yi, yi], color=C_M2B, lw=3 if main else 2, solid_capstyle="round", zorder=3)
    ax.plot(d, yi, "o", ms=11 if main else 8, color=C_M2B, mec=SURF, mew=2, zorder=4)
    ax.text(col, yi, sg(d), va="center", ha="right", fontsize=10.5, fontweight="bold" if main else "normal")
ax.set_yticks(y, [r[0] for r in rows], fontsize=10.5)
for t, r in zip(ax.get_yticklabels(), rows):
    if r[4]: t.set_fontweight("bold")
ax.set_xlim(lo, col + 0.01); ax.set_xticks([t for t in np.arange(-0.2, hi + 0.01, 0.1) if t >= lo])
ax.text(lo + 0.005, len(rows) - 0.35, "M2b lebih rendah", fontsize=9, color=INK2)
ax.text(col, len(rows) - 0.35, "Selisih", fontsize=9, color=INK2, ha="right")
ax.set_xlabel("Selisih F1 (M2b dikurangi M0). Kanan dari garis nol = M2b lebih tinggi"); clean(ax, "x")
melewati = all(r[2] <= 0 <= r[3] for r in rows)
finish(fig, "fig3_selisih_f1.png", "Semua kisaran masih melewati nol, jadi belum terbukti" if melewati else "Selisih F1 dan kisaran kepercayaan 95%",
       "Titik = selisih F1, garis = kisaran 95% (bootstrap per template).")

# ---------- 4. kartu skor ----------
keys = [("Recall", "Recall\n(tertangkap)"), ("Precision", "Precision\n(tanda yang benar)"), ("Specificity", "Specificity\n(Normal dikenali)"),
        ("BalancedAcc", "Balanced\nAccuracy"), ("F1", "F1"), ("MCC", "MCC")]
m0v = [e0[k_] for k_, _ in keys]; m2v = [e2[k_] for k_, _ in keys]
fig, ax = plt.subplots(figsize=(10, 5)); x = np.arange(len(keys)); w = 0.36
b0 = ax.bar(x - w/2 - 0.01, m0v, w, color=C_M0, zorder=3, label="Metode Wu (M0)")
b2 = ax.bar(x + w/2 + 0.01, m2v, w, color=C_M2B, zorder=3, label="M2b (model saya)")
for bs in (b0, b2):
    for r in bs:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.012, c(r.get_height()), ha="center", fontsize=8.5)
ax.set_xticks(x, [n for _, n in keys], fontsize=10); ax.set_ylim(0, 1.05); clean(ax)
ax.legend(frameon=False, loc="upper right", ncol=2, fontsize=10.5)
finish(fig, "fig4_kartu_skor.png", "Kartu skor (data uji baru, label v2)",
       "Makin tinggi makin bagus. Belum ada kisaran kepercayaan untuk Balanced Accuracy dan MCC, jadi dibaca sebagai indikasi.")

# ---------- 5. tabel tebakan benar dan salah ----------
cmap = LinearSegmentedColormap.from_list("blu", ["#f4f8fd", "#2a78d6"])
fig, axs = plt.subplots(1, 2, figsize=(9.5, 4.6))
for ax, (t, e) in zip(axs, [("Metode Wu (M0)", e0), ("M2b (model saya)", e2)]):
    m = np.array([[e["TN"], e["FP"]], [e["FN"], e["TP"]]]).astype(int)
    ax.imshow(m, cmap=cmap, vmin=0, vmax=m.max())
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(m[i, j]), ha="center", va="center", fontsize=20, fontweight="bold",
                    color="white" if m[i, j] > m.max() * 0.5 else INK)
    ax.set_xticks([0, 1], ["Ditandai\nNormal", "Ditandai\nRegression"], fontsize=10)
    ax.set_yticks([0, 1], ["Aslinya\nNormal", "Aslinya\nRegression"], fontsize=10)
    ax.set_title(t, fontsize=12, loc="left"); ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
finish(fig, "fig5_tabel_tebakan.png", "Tabel tebakan benar dan salah (data uji baru, label v2)",
       "Kiri atas = Normal benar. Kanan bawah = Regression tertangkap. Kanan atas = salah alarm. Kiri bawah = Regression lolos.", top=0.92)
print("\nTersimpan di:", OUT)

## Ablasi dan baseline tambahan


In [ ]:
# ============================================================
# ABLASI FORMAL + BASELINE TAMBAHAN  (latih di data LAMA, uji di data BARU seed 2026)
# ------------------------------------------------------------
# Tujuan: menjawab dua pertanyaan evaluasi.
#  (1) Bagian variabel mana yang membuat M2b lebih baik?  -> ablasi M1, M2, M1+loc, M2b
#  (2) Apakah XGBoost memang perlu?                        -> logistic regression, random forest, aturan biaya
#
# ATURAN (ditulis sebelum melihat hasil):
#  - Semua model dilatih HANYA di data lama. Pengaturan tetap, tanpa tuning. Data baru dipakai untuk menguji.
#  - Perbandingan utama memakai ANGGARAN SAMA (top-k, k = jumlah tanda M0), jadi tidak bergantung pada threshold.
#  - Metrik utama: F1 pada anggaran sama; selisih terhadap M0 dan antar tahap ablasi dengan CI 95% bootstrap per template.
#  - Label utama v2; label v1 hanya pelengkap.
#  - Ini analisis DESKRIPTIF pada batch 2026 yang sudah pernah dilihat. Bukan uji konfirmasi.
#  - Cell ini menolak membaca batch confirm2027 sebelum uji konfirmasinya selesai (supaya tidak "mengintip").
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score, precision_score,
                             recall_score, matthews_corrcoef)

DRIVE_ROOT = Path("/content/drive/MyDrive/QPR_SF10_Research")
OLD_ROOT = DRIVE_ROOT / "incremental"
NEW_ROOT = INCREMENTAL_ROOT if "INCREMENTAL_ROOT" in globals() and "fresh" in str(INCREMENTAL_ROOT) \
    else DRIVE_ROOT / "fresh" / "incremental_fresh"
if "confirm2027" in str(NEW_ROOT):
    assert (DRIVE_ROOT / "confirm2027" / "confirm2027_RESULT.json").exists(), \
        "Uji konfirmasi 2027 belum selesai. Jangan jalankan analisis ini di batch 2027 sebelum itu."
OLD_FEAT = OLD_ROOT / "incremental_m0_m2b_features.csv"
OLD_V2 = OLD_ROOT / "remeasure" / "remeasure_labels_v2.csv"
NEW_FEAT = NEW_ROOT / "incremental_m0_m2b_features.csv"
NEW_V2 = NEW_ROOT / "remeasure" / "remeasure_labels_v2.csv"
OUT_CSV = NEW_ROOT / "remeasure" / "ablasi_baseline_hasil.csv"
SEEDS = [42, 43, 44, 45, 46]
N_BOOT, BOOT_SEED = 2000, 2026
for p in [OLD_FEAT, OLD_V2, NEW_FEAT, NEW_V2]:
    assert p.exists(), f"File tidak ditemukan: {p}"

old = pd.read_csv(OLD_FEAT).merge(pd.read_csv(OLD_V2)[["sample_id", "label_v1", "label_v2"]], on="sample_id", how="inner")
new = pd.read_csv(NEW_FEAT)
new["label_v1"] = new["label"].astype(int)
new = new.merge(pd.read_csv(NEW_V2)[["sample_id", "label_v2"]], on="sample_id", how="left")

forbidden = {"before_elapsed_ms", "after_elapsed_ms", "runtime_ratio", "label",
             "label_v1", "label_v2", "kepastian_v2", "label_v1_file"}
def cols_with(prefixes, exact=()):
    return [c for c in old.columns if c not in forbidden and not c.startswith("m0_") and "elapsed_ms" not in c
            and (c.startswith(tuple(prefixes)) or c in exact) and pd.api.types.is_numeric_dtype(old[c])]
BASE = ["before_", "candidate_", "delta_"]
GROUPS = {
    "M1  (bentuk plan saja)":                 cols_with(BASE, ("estimated_cost_ratio",)),
    "M2  (M1 + CE seluruh plan)":             cols_with(BASE + ["ce_"], ("estimated_cost_ratio",)),
    "M1+loc (M1 + CE bagian berubah)":        cols_with(BASE + ["loc_"], ("estimated_cost_ratio",)),
    "M2b (M1 + CE seluruh + CE berubah)":     cols_with(BASE + ["ce_", "loc_"], ("estimated_cost_ratio",)),
}
ALL = GROUPS["M2b (M1 + CE seluruh + CE berubah)"]
assert all(c in new.columns for c in ALL), "Ada fitur yang tidak ada di data baru"
print("Jumlah fitur per model:", {k: len(v) for k, v in GROUPS.items()})
assert len(ALL) == len(cols_with(BASE + ["ce_", "loc_"], ("estimated_cost_ratio",)))

def prep(d, cols):
    return d[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).reset_index(drop=True)

def xgb(y, seed):
    pos = int((y == 1).sum()); neg = int((y == 0).sum())
    return XGBClassifier(objective="binary:logistic", n_estimators=200, learning_rate=0.05, max_depth=3,
                         min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, reg_alpha=0.0,
                         scale_pos_weight=neg / max(pos, 1), eval_metric="logloss", tree_method="hist",
                         n_jobs=2, random_state=seed, verbosity=0)
def logreg(y, seed):
    slog = FunctionTransformer(lambda a: np.sign(a) * np.log1p(np.abs(a)))
    return make_pipeline(slog, StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000))
def rforest(y, seed):
    return RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=2, class_weight="balanced_subsample",
                                  n_jobs=2, random_state=seed)

MODELS = [(k, v, xgb) for k, v in GROUPS.items()] + [
    ("LogReg (semua fitur)", ALL, logreg), ("RandomForest (semua fitur)", ALL, rforest)]

def f1_fast(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

def topk(prob, k):
    p = np.zeros(len(prob), dtype=int); p[np.argsort(-prob, kind="stable")[:k]] = 1; return p

def pick_threshold(builder, X, y, g):
    k = min(3, max(len(set(g[y == 1])), 2))
    inner = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=123)
    prob = np.zeros(len(y))
    for tr, va in inner.split(X, y, g):
        if len(set(y[tr])) < 2: continue
        m = builder(y[tr], 42); m.fit(X.iloc[tr], y[tr]); prob[va] = m.predict_proba(X.iloc[va])[:, 1]
    best, bt = None, 0.5
    for t in np.linspace(0.05, 0.90, 86):
        p = (prob >= t).astype(int)
        s = (f1_score(y, p, zero_division=0), matthews_corrcoef(y, p), recall_score(y, p, zero_division=0),
             precision_score(y, p, zero_division=0))
        if best is None or s > best: best, bt = s, float(t)
    return bt

def boot_f1(y, preds, groups):
    """satu set resample per template dipakai untuk semua model -> perbandingan berpasangan yang adil"""
    rng = np.random.default_rng(BOOT_SEED)
    ug = np.unique(groups); idx = {g: np.where(groups == g)[0] for g in ug}
    out = {k: [] for k in preds}
    for _ in range(N_BOOT):
        ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)])
        for k, p in preds.items(): out[k].append(f1_fast(y[ii], p[ii]))
    return {k: np.array(v) for k, v in out.items()}

def dci(b, a, c):  # selisih F1 (a minus c) beserta CI 95%
    d = b[a] - b[c]; return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

all_rows = []
for lab, tr_lab in [("label_v2", "label_v2"), ("label_v1", "label_v1")]:
    y_tr = old[tr_lab].astype(int).to_numpy(); g_old = old["template_id"].astype(str).to_numpy()
    valid = new[lab].notna().to_numpy()
    d = new.loc[valid].reset_index(drop=True)
    y = d[lab].astype(int).to_numpy(); g = d["template_id"].astype(str).to_numpy(); p0 = d["m0_prediction"].astype(int).to_numpy()
    k = int(p0.sum()); n = len(y)
    print("\n" + "=" * 78)
    print(f"LABEL UJI {lab} | n={n} | Regression={int(y.sum())} | M0 menandai k={k}")
    print("=" * 78)
    if y.sum() < 3 or k == 0:
        print("  Data tidak cukup."); continue

    preds_mb, probs, seed_f1 = {"M0 (Wu)": p0}, {}, {}
    for name, cols, builder in MODELS:
        Xo, Xn = prep(old, cols), prep(d, cols)
        fs = []
        for s in SEEDS:
            m = builder(y_tr, s); m.fit(Xo, y_tr); pr = m.predict_proba(Xn)[:, 1]
            if s == SEEDS[0]: probs[name] = pr; preds_mb[name] = topk(pr, k)
            fs.append(f1_fast(y, topk(pr, k)))
            if builder is logreg: break              # LogReg tidak acak
        seed_f1[name] = fs
    # aturan sederhana: urutkan menurut estimated_cost_ratio (makin besar makin berisiko)
    if "estimated_cost_ratio" in d.columns:
        sc = d["estimated_cost_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
        probs["Aturan: estimated_cost_ratio"] = sc; preds_mb["Aturan: estimated_cost_ratio"] = topk(sc, k)
        seed_f1["Aturan: estimated_cost_ratio"] = [f1_fast(y, preds_mb["Aturan: estimated_cost_ratio"])]
    bt = boot_f1(y, preds_mb, g)

    print(f"\nA. ANGGARAN SAMA (top-{k}, tanpa threshold)")
    print(f"{'Model':40s} {'TP':>3s} {'FP':>4s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s} {'F1 5 seed':>16s}  selisih F1 vs M0 [CI 95%]   PR-AUC ROC-AUC")
    for name, p in preds_mb.items():
        tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
        f1v = f1_fast(y, p); sf = seed_f1.get(name, [f1v])
        if name == "M0 (Wu)":
            print(f"{name:40s} {tp:3d} {fp:4d} {tp / max(int(p.sum()), 1):6.3f} {tp / y.sum():6.3f} {f1v:6.3f} {'-':>16s}  {'(pembanding)':>26s}"); continue
        lo, hi = dci(bt, name, "M0 (Wu)")
        pa = average_precision_score(y, probs[name]); ra = roc_auc_score(y, probs[name])
        print(f"{name:40s} {tp:3d} {fp:4d} {tp / max(int(p.sum()), 1):6.3f} {tp / y.sum():6.3f} {f1v:6.3f} "
              f"{np.mean(sf):6.3f}±{np.std(sf):5.3f}   {f1v - f1_fast(y, p0):+.3f} [{lo:+.3f},{hi:+.3f}]  {pa:6.3f} {ra:7.3f}")
        all_rows.append({"label": lab, "analisis": "anggaran_sama", "model": name, "k": k, "TP": tp, "FP": fp,
                         "F1": f1v, "F1_seed_mean": float(np.mean(sf)), "dF1_vs_M0": f1v - f1_fast(y, p0),
                         "CI_low": lo, "CI_high": hi, "PR_AUC": pa, "ROC_AUC": ra})

    full = "M2b (M1 + CE seluruh + CE berubah)"
    print("\nB. KONTRAS ABLASI (anggaran sama): apa yang ditambah M2b dibanding tahap sebelumnya")
    for other in ["M1  (bentuk plan saja)", "M2  (M1 + CE seluruh plan)", "M1+loc (M1 + CE bagian berubah)",
                  "LogReg (semua fitur)", "RandomForest (semua fitur)"]:
        lo, hi = dci(bt, full, other)
        print(f"  M2b - {other:36s} = {f1_fast(y, preds_mb[full]) - f1_fast(y, preds_mb[other]):+.3f}  [{lo:+.3f}, {hi:+.3f}]"
              f"{'   <- CI tidak memuat 0' if lo > 0 or hi < 0 else ''}")
        all_rows.append({"label": lab, "analisis": "kontras", "model": f"M2b - {other}",
                         "dF1_vs_M0": f1_fast(y, preds_mb[full]) - f1_fast(y, preds_mb[other]), "CI_low": lo, "CI_high": hi})

    print("\nC. PAKAI THRESHOLD (dipilih hanya dari data lama, CV bersarang) - sama seperti evaluasi terkunci")
    print(f"{'Model':40s} {'thr':>5s} {'ditandai':>8s} {'TP':>3s} {'FP':>4s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s} {'selisih vs M0':>14s}")
    f0 = f1_fast(y, p0)
    print(f"{'M0 (Wu)':40s} {'-':>5s} {k:8d} {int(((y == 1) & (p0 == 1)).sum()):3d} {int(((y == 0) & (p0 == 1)).sum()):4d} "
          f"{precision_score(y, p0, zero_division=0):6.3f} {recall_score(y, p0, zero_division=0):6.3f} {f0:6.3f}")
    for name, cols, builder in MODELS:
        Xo, Xn = prep(old, cols), prep(d, cols)
        thr = pick_threshold(builder, Xo, y_tr, g_old)
        p = (probs[name] >= thr).astype(int)
        tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
        print(f"{name:40s} {thr:5.2f} {int(p.sum()):8d} {tp:3d} {fp:4d} {precision_score(y, p, zero_division=0):6.3f} "
              f"{recall_score(y, p, zero_division=0):6.3f} {f1_fast(y, p):6.3f} {f1_fast(y, p) - f0:+14.3f}")
        all_rows.append({"label": lab, "analisis": "threshold", "model": name, "k": int(p.sum()), "TP": tp, "FP": fp,
                         "F1": f1_fast(y, p), "dF1_vs_M0": f1_fast(y, p) - f0})

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(all_rows).to_csv(OUT_CSV, index=False)
print("\nTersimpan:", OUT_CSV)
print("\nCARA BACA:")
print("  - Bagian A menjawab 'apakah XGBoost perlu?': kalau LogReg/RandomForest/aturan biaya sama bagusnya, laporkan apa adanya.")
print("  - Bagian B menjawab 'variabel mana yang berperan?': M2b - M1+loc dan M2b - M2 menunjukkan nilai tambah CE berubah vs CE seluruh plan.")
print("  - CI yang memuat 0 = indikasi saja. Ini batch 2026 yang sudah pernah dilihat; konfirmasi tetap dari uji seed 2027.")

## Uji silang dua arah dan gabungan peringkat


In [ ]:
# ============================================================
# UJI SILANG DUA ARAH + KANDIDAT PERBAIKAN  (data lama + data baru seed 2026)
# ------------------------------------------------------------
# Tujuan:
#  (1) Menambah daya uji. Evaluasi satu arah hanya memuat 30 query Regression. Di sini dua arah dipakai:
#        Arah 1: latih di data LAMA  -> uji di data BARU   (sama dengan uji terkunci)
#        Arah 2: latih di data BARU  -> uji di data LAMA   (arah sebaliknya)
#      Setiap transisi diuji oleh model yang TIDAK pernah melihatnya, jadi ~650 transisi dan jumlah Regression
#      hampir dua kali lipat. CI diambil dengan bootstrap per template pada gabungan dua arah.
#  (2) Mencoba kandidat perbaikan yang masuk akal dari hasil ablasi:
#        - Gabungan peringkat (XGBoost struktur plan + XGBoost lengkap + Random forest)
#        - XGBoost + sinyal Detektor Pola Wu sebagai satu variabel tambahan (hibrida)
#
# ATURAN (ditulis sebelum melihat hasil):
#  - KANDIDAT UTAMA yang dinilai: "Gabungan peringkat". Yang lain dilaporkan sebagai pelengkap, bukan untuk dipilih-pilih.
#  - Semua kandidat DILAPORKAN, termasuk yang jelek. Tidak ada yang disembunyikan.
#  - Pengaturan model tetap (sama dengan ablasi). Probabilitas dirata-rata dari 5 seed supaya tidak bergantung satu seed.
#  - Perbandingan memakai ANGGARAN SAMA: tiap arah dibatasi menandai sebanyak yang ditandai Detektor Pola Wu pada
#    set uji arah itu. Tidak ada threshold yang disetel.
#  - Metrik utama: selisih F1 terhadap Detektor Pola Wu, CI 95% bootstrap per template (2000 ulangan). Label utama v2.
#  - Ini analisis DESKRIPTIF pada data yang sudah pernah dilihat. Ia memperkuat atau melemahkan, bukan pembuktian akhir.
#  - Cell ini menolak dijalankan pada batch konfirmasi.
# ============================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import rankdata

DRIVE_ROOT = Path(os.environ.get("QPR_ROOT", "/content/drive/MyDrive/QPR_SF10_Research"))
OLD_ROOT = DRIVE_ROOT / "incremental"
NEW_ROOT = DRIVE_ROOT / "fresh" / "incremental_fresh"
OLD_FEAT = OLD_ROOT / "incremental_m0_m2b_features.csv"
OLD_V2 = OLD_ROOT / "remeasure" / "remeasure_labels_v2.csv"
NEW_FEAT = NEW_ROOT / "incremental_m0_m2b_features.csv"
NEW_V2 = NEW_ROOT / "remeasure" / "remeasure_labels_v2.csv"
OUT_CSV = NEW_ROOT / "remeasure" / "perbaikan_gabungan_hasil.csv"
SEEDS = [42, 43, 44, 45, 46]
N_BOOT, BOOT_SEED = 2000, 2026
for p in [OLD_FEAT, OLD_V2, NEW_FEAT, NEW_V2]:
    assert p.exists(), f"File tidak ditemukan: {p}"

old = pd.read_csv(OLD_FEAT).merge(pd.read_csv(OLD_V2)[["sample_id", "label_v1", "label_v2"]], on="sample_id", how="inner")
new = pd.read_csv(NEW_FEAT)
new["label_v1"] = new["label"].astype(int)
new = new.merge(pd.read_csv(NEW_V2)[["sample_id", "label_v2"]], on="sample_id", how="left")

forbidden = {"before_elapsed_ms", "after_elapsed_ms", "runtime_ratio", "label",
             "label_v1", "label_v2", "kepastian_v2", "label_v1_file"}
def cols_with(prefixes, exact=()):
    return [c for c in old.columns if c not in forbidden and not c.startswith("m0_") and "elapsed_ms" not in c
            and (c.startswith(tuple(prefixes)) or c in exact) and pd.api.types.is_numeric_dtype(old[c])]
BASE = ["before_", "candidate_", "delta_"]
M1_COLS = cols_with(BASE, ("estimated_cost_ratio",))
ALL_COLS = cols_with(BASE + ["ce_", "loc_"], ("estimated_cost_ratio",))
assert "m0_prediction" in old.columns and "m0_prediction" in new.columns, "m0_prediction tidak ada"
assert all(c in new.columns for c in ALL_COLS), "Ada fitur yang tidak ada di salah satu data"
print(f"Fitur: struktur plan={len(M1_COLS)}, lengkap={len(ALL_COLS)}")

def prep(d, cols):
    X = d[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).reset_index(drop=True)
    return X

def xgb(y, seed):
    pos = int((y == 1).sum()); neg = int((y == 0).sum())
    return XGBClassifier(objective="binary:logistic", n_estimators=200, learning_rate=0.05, max_depth=3,
                         min_child_weight=1, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, reg_alpha=0.0,
                         scale_pos_weight=neg / max(pos, 1), eval_metric="logloss", tree_method="hist",
                         n_jobs=2, random_state=seed, verbosity=0)
def rforest(y, seed):
    return RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=2, class_weight="balanced_subsample",
                                  n_jobs=2, random_state=seed)

def avg_prob(builder, tr, te, cols, y_tr, add_m0=False):
    Xtr, Xte = prep(tr, cols), prep(te, cols)
    if add_m0:
        Xtr["wu_flag"] = tr["m0_prediction"].astype(int).to_numpy(); Xte["wu_flag"] = te["m0_prediction"].astype(int).to_numpy()
    out = np.zeros(len(te))
    for s in SEEDS:
        m = builder(y_tr, s); m.fit(Xtr, y_tr); out += m.predict_proba(Xte)[:, 1]
    return out / len(SEEDS)

def rank01(p):
    return (rankdata(p) - 1) / max(len(p) - 1, 1)

def f1_fast(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
def topk(prob, k):
    p = np.zeros(len(prob), dtype=int); p[np.argsort(-prob, kind="stable")[:k]] = 1; return p

NAMES = ["Detektor Pola Wu",
         "XGBoost Struktur Plan",
         "XGBoost Kardinalitas Lokal (lengkap)",
         "Random forest",
         "Gabungan peringkat (KANDIDAT UTAMA)",
         "Hibrida: XGBoost Struktur Plan + sinyal Wu",
         "Hibrida: XGBoost lengkap + sinyal Wu"]

def one_direction(tr, te, lab, mult=1.0):
    y_tr = tr[lab].astype(int).to_numpy()
    y = te[lab].astype(int).to_numpy(); g = te["template_id"].astype(str).to_numpy()
    p0 = te["m0_prediction"].astype(int).to_numpy(); k0 = int(p0.sum())
    pr = {}
    pr["XGBoost Struktur Plan"] = avg_prob(xgb, tr, te, M1_COLS, y_tr)
    pr["XGBoost Kardinalitas Lokal (lengkap)"] = avg_prob(xgb, tr, te, ALL_COLS, y_tr)
    pr["Random forest"] = avg_prob(rforest, tr, te, ALL_COLS, y_tr)
    pr["Gabungan peringkat (KANDIDAT UTAMA)"] = (rank01(pr["XGBoost Struktur Plan"]) +
        rank01(pr["XGBoost Kardinalitas Lokal (lengkap)"]) + rank01(pr["Random forest"])) / 3.0
    pr["Hibrida: XGBoost Struktur Plan + sinyal Wu"] = avg_prob(xgb, tr, te, M1_COLS, y_tr, add_m0=True)
    pr["Hibrida: XGBoost lengkap + sinyal Wu"] = avg_prob(xgb, tr, te, ALL_COLS, y_tr, add_m0=True)
    return y, g, p0, k0, pr

def boot(y, g, preds):
    rng = np.random.default_rng(BOOT_SEED)
    ug = np.unique(g); idx = {t: np.where(g == t)[0] for t in ug}
    out = {k: np.empty(N_BOOT) for k in preds}
    for b in range(N_BOOT):
        ii = np.concatenate([idx[t] for t in rng.choice(ug, len(ug), replace=True)])
        for k, p in preds.items(): out[k][b] = f1_fast(y[ii], p[ii])
    return out
def ci(b, a, c):
    d = b[a] - b[c]; return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

rows = []
for lab in ["label_v2", "label_v1"]:
    o = old[old[lab].notna()].reset_index(drop=True)
    n_ = new[new[lab].notna()].reset_index(drop=True)
    print("\n" + "=" * 86)
    print(f"{lab.upper()}  |  data lama n={len(o)} (Regression {int(o[lab].sum())})  |  data baru n={len(n_)} (Regression {int(n_[lab].sum())})")
    print("=" * 86)
    d1 = one_direction(o, n_, lab)   # lama -> baru
    d2 = one_direction(n_, o, lab)   # baru -> lama
    for nm, d in [("Arah 1 (latih LAMA, uji BARU)", d1), ("Arah 2 (latih BARU, uji LAMA)", d2)]:
        y, g, p0, k0, pr = d
        print(f"\n{nm}: n={len(y)}, Regression={int(y.sum())}, Wu menandai k={k0}")
        print(f"  {'Model':46s} {'TP':>3s} {'FP':>4s} {'F1':>6s}")
        print(f"  {'Detektor Pola Wu':46s} {int(((y==1)&(p0==1)).sum()):3d} {int(((y==0)&(p0==1)).sum()):4d} {f1_fast(y, p0):6.3f}")
        for k_, v in pr.items():
            p = topk(v, k0); print(f"  {k_:46s} {int(((y==1)&(p==1)).sum()):3d} {int(((y==0)&(p==1)).sum()):4d} {f1_fast(y, p):6.3f}")
    # gabungan dua arah
    y = np.concatenate([d1[0], d2[0]]); g = np.concatenate([d1[1], d2[1]])
    preds = {"Detektor Pola Wu": np.concatenate([d1[2], d2[2]])}
    for k_ in NAMES[1:]:
        preds[k_] = np.concatenate([topk(d1[4][k_], d1[3]), topk(d2[4][k_], d2[3])])
    b = boot(y, g, preds)
    f0 = f1_fast(y, preds["Detektor Pola Wu"])
    print(f"\nGABUNGAN DUA ARAH (anggaran sama per arah): n={len(y)}, Regression={int(y.sum())}, ditandai Wu={int(preds['Detektor Pola Wu'].sum())}")
    print(f"  {'Model':46s} {'TP':>3s} {'FP':>4s} {'Prec':>6s} {'Rec':>6s} {'F1':>6s}   selisih F1 vs Wu [CI 95%]")
    for k_, p in preds.items():
        tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum()); f = f1_fast(y, p)
        if k_ == "Detektor Pola Wu":
            print(f"  {k_:46s} {tp:3d} {fp:4d} {tp/max(tp+fp,1):6.3f} {tp/max(int(y.sum()),1):6.3f} {f:6.3f}   (pembanding)"); continue
        lo, hi = ci(b, k_, "Detektor Pola Wu")
        flag = "   <- CI tidak memuat 0" if lo > 0 else ""
        print(f"  {k_:46s} {tp:3d} {fp:4d} {tp/max(tp+fp,1):6.3f} {tp/max(int(y.sum()),1):6.3f} {f:6.3f}   {f-f0:+.3f} [{lo:+.3f},{hi:+.3f}]{flag}")
        rows.append({"label": lab, "model": k_, "TP": tp, "FP": fp, "F1": f, "dF1_vs_Wu": f - f0, "CI_low": lo, "CI_high": hi,
                     "n": len(y), "Regression": int(y.sum())})
    print("\n  Kontras terhadap XGBoost Kardinalitas Lokal (lengkap), model yang diusulkan semula:")
    base = "XGBoost Kardinalitas Lokal (lengkap)"
    for k_ in NAMES[1:]:
        if k_ == base: continue
        lo, hi = ci(b, k_, base)
        print(f"    {k_:46s} = {f1_fast(y, preds[k_]) - f1_fast(y, preds[base]):+.3f} [{lo:+.3f},{hi:+.3f}]"
              f"{'   <- CI tidak memuat 0' if lo > 0 or hi < 0 else ''}")
    # sapuan anggaran (deskriptif): F1 jika anggaran dikali 0,5 / 1 / 1,5 / 2 dari jumlah tanda Wu
    print("\n  Sapuan anggaran (deskriptif): F1 gabungan dua arah bila jumlah tanda = kelipatan dari jumlah tanda Wu")
    print(f"  {'Model':46s} " + " ".join(f"{m:>6.1f}x" for m in [0.5, 1.0, 1.5, 2.0]))
    for k_ in NAMES[1:]:
        vals = []
        for m in [0.5, 1.0, 1.5, 2.0]:
            pp = np.concatenate([topk(d1[4][k_], int(round(m * d1[3]))), topk(d2[4][k_], int(round(m * d2[3])))])
            vals.append(f1_fast(y, pp))
        print(f"  {k_:46s} " + " ".join(f"{v:7.3f}" for v in vals))

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print("\nTersimpan:", OUT_CSV)
print("\nCARA BACA:")
print("  - Bagian GABUNGAN DUA ARAH adalah angka utama. Lihat baris 'Gabungan peringkat' (kandidat utama).")
print("  - Kalau CI selisih F1 terhadap Wu tidak memuat 0, itu penguatan bukti. Kalau masih memuat 0, dilaporkan: belum terbukti.")
print("  - Sapuan anggaran hanya deskriptif (menunjukkan model mana yang lebih baik di berbagai jumlah tanda).")

## Figur dan metrik tambahan


In [ ]:
# ============================================================
# FIGUR DAN METRIK TAMBAHAN UNTUK MANUSKRIP
# ------------------------------------------------------------
# Menghasilkan (tanpa tuning, pengaturan sama dengan ablasi dan uji dua arah):
#  1. Tabel metrik multi-dimensi: Precision, Recall, F1, PR-AUC, ROC-AUC, Brier Score, latensi
#  2. Kurva Precision-Recall (arah 1: latih LAMA uji BARU, dan arah 2: sebaliknya)
#  3. Confusion matrix (Detektor Pola Wu vs Gabungan peringkat, gabungan dua arah, anggaran sama)
#  4. Pentingnya variabel XGBoost (gain per variabel dan per kelompok variabel)
#  5. Kurva konvergensi XGBoost (log-loss latih vs validasi, validasi = pecahan template data lama)
# Semua figur disimpan ke Drive: QPR_SF10_Research/artikel_gambar/ , tabelnya ke metrik_tambahan.csv
# Label utama: label_v2. Cell ini menolak dijalankan pada batch konfirmasi.
# ============================================================
import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss, precision_recall_curve,
                             confusion_matrix)
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import rankdata

DRIVE_ROOT = Path(os.environ.get("QPR_ROOT", "/content/drive/MyDrive/QPR_SF10_Research"))
OLD_ROOT = DRIVE_ROOT / "incremental"
NEW_ROOT = DRIVE_ROOT / "fresh" / "incremental_fresh"
OUT_DIR = DRIVE_ROOT / "artikel_gambar"; OUT_DIR.mkdir(parents=True, exist_ok=True)
OLD_FEAT = OLD_ROOT / "incremental_m0_m2b_features.csv"
OLD_V2 = OLD_ROOT / "remeasure" / "remeasure_labels_v2.csv"
NEW_FEAT = NEW_ROOT / "incremental_m0_m2b_features.csv"
NEW_V2 = NEW_ROOT / "remeasure" / "remeasure_labels_v2.csv"
SEEDS = [42, 43, 44, 45, 46]
LAB = "label_v2"
for p in [OLD_FEAT, OLD_V2, NEW_FEAT, NEW_V2]:
    assert p.exists(), f"File tidak ditemukan: {p}"

old = pd.read_csv(OLD_FEAT).merge(pd.read_csv(OLD_V2)[["sample_id", "label_v1", "label_v2"]], on="sample_id", how="inner")
new = pd.read_csv(NEW_FEAT)
new["label_v1"] = new["label"].astype(int)
new = new.merge(pd.read_csv(NEW_V2)[["sample_id", "label_v2"]], on="sample_id", how="left")
old = old[old[LAB].notna()].reset_index(drop=True); new = new[new[LAB].notna()].reset_index(drop=True)

forbidden = {"before_elapsed_ms", "after_elapsed_ms", "runtime_ratio", "label", "label_v1", "label_v2", "kepastian_v2", "label_v1_file"}
def cols_with(prefixes, exact=()):
    return [c for c in old.columns if c not in forbidden and not c.startswith("m0_") and "elapsed_ms" not in c
            and (c.startswith(tuple(prefixes)) or c in exact) and pd.api.types.is_numeric_dtype(old[c])]
BASE = ["before_", "candidate_", "delta_"]
M1_COLS = cols_with(BASE, ("estimated_cost_ratio",))
ALL_COLS = cols_with(BASE + ["ce_", "loc_"], ("estimated_cost_ratio",))
assert all(c in new.columns for c in ALL_COLS)

def prep(d, cols): return d[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).reset_index(drop=True)
def xgb(y, seed, **extra):
    pos = int((y == 1).sum()); neg = int((y == 0).sum())
    return XGBClassifier(objective="binary:logistic", n_estimators=200, learning_rate=0.05, max_depth=3, min_child_weight=1,
                         subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, reg_alpha=0.0, scale_pos_weight=neg / max(pos, 1),
                         eval_metric="logloss", tree_method="hist", n_jobs=2, random_state=seed, verbosity=0, **extra)
def rforest(y, seed):
    return RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=2, class_weight="balanced_subsample",
                                  n_jobs=2, random_state=seed)
def rank01(p): return (rankdata(p) - 1) / max(len(p) - 1, 1)
def topk(prob, k):
    p = np.zeros(len(prob), dtype=int); p[np.argsort(-prob, kind="stable")[:k]] = 1; return p
def f1_of(tp, fp, fn): return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

NM = {"wu": "Detektor Pola Wu", "m1": "XGBoost Struktur Plan", "m2b": "XGBoost Kardinalitas Lokal",
      "rf": "Random forest", "ens": "Gabungan peringkat"}

def direction(tr, te):
    y_tr = tr[LAB].astype(int).to_numpy(); y = te[LAB].astype(int).to_numpy()
    out = {"y": y, "p0": te["m0_prediction"].astype(int).to_numpy(), "prob": {}, "lat_ms": {}, "fit_s": {}}
    specs = [("m1", xgb, M1_COLS), ("m2b", xgb, ALL_COLS), ("rf", rforest, ALL_COLS)]
    for key, builder, cols in specs:
        Xtr, Xte = prep(tr, cols), prep(te, cols)
        acc = np.zeros(len(te)); t_fit = 0.0; t_pred = 0.0
        for s in SEEDS:
            t0 = time.perf_counter(); m = builder(y_tr, s); m.fit(Xtr, y_tr); t_fit += time.perf_counter() - t0
            t0 = time.perf_counter(); acc += m.predict_proba(Xte)[:, 1]; t_pred += time.perf_counter() - t0
        out["prob"][key] = acc / len(SEEDS)
        out["fit_s"][key] = t_fit / len(SEEDS); out["lat_ms"][key] = 1000 * (t_pred / len(SEEDS)) / len(te)
    out["prob"]["ens"] = (rank01(out["prob"]["m1"]) + rank01(out["prob"]["m2b"]) + rank01(out["prob"]["rf"])) / 3.0
    out["lat_ms"]["ens"] = out["lat_ms"]["m1"] + out["lat_ms"]["m2b"] + out["lat_ms"]["rf"]   # tiga model dijalankan berurutan
    out["fit_s"]["ens"] = out["fit_s"]["m1"] + out["fit_s"]["m2b"] + out["fit_s"]["rf"]
    out["k"] = int(out["p0"].sum())
    return out

D1 = direction(old, new)   # arah 1: latih LAMA, uji BARU
D2 = direction(new, old)   # arah 2: latih BARU, uji LAMA
KEYS = ["m1", "m2b", "rf", "ens"]

# ---------- 1. TABEL METRIK ----------
rows = []
y_all = np.concatenate([D1["y"], D2["y"]])
def pooled_pred(key):
    return np.concatenate([topk(D1["prob"][key], D1["k"]), topk(D2["prob"][key], D2["k"])]) if key != "wu" else np.concatenate([D1["p0"], D2["p0"]])
for key in ["wu"] + KEYS:
    p = pooled_pred(key)
    tp = int(((y_all == 1) & (p == 1)).sum()); fp = int(((y_all == 0) & (p == 1)).sum()); fn = int(((y_all == 1) & (p == 0)).sum())
    tn = int(((y_all == 0) & (p == 0)).sum())
    r = {"model": NM[key], "TP": tp, "FP": fp, "FN": fn, "TN": tn,
         "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1), "F1": f1_of(tp, fp, fn),
         "Akurasi": (tp + tn) / len(y_all)}
    if key != "wu":
        pa, ra, br, lat = [], [], [], []
        for D in (D1, D2):
            pa.append(average_precision_score(D["y"], D["prob"][key])); ra.append(roc_auc_score(D["y"], D["prob"][key]))
            if key != "ens": br.append(brier_score_loss(D["y"], D["prob"][key]))
        r["PR_AUC_arah1"], r["PR_AUC_arah2"] = pa; r["ROC_AUC_arah1"], r["ROC_AUC_arah2"] = ra
        r["PR_AUC_rata"], r["ROC_AUC_rata"] = float(np.mean(pa)), float(np.mean(ra))
        r["Brier_rata"] = float(np.mean(br)) if br else np.nan
        r["Latensi_ms_per_transisi"] = float(np.mean([D1["lat_ms"][key], D2["lat_ms"][key]]))
        r["Waktu_latih_detik"] = float(np.mean([D1["fit_s"][key], D2["fit_s"][key]]))
    rows.append(r)
tab = pd.DataFrame(rows)
prev = [D["y"].mean() for D in (D1, D2)]
print("Laju dasar Regression per arah:", [f"{v:.3f}" for v in prev], "| Brier acuan (selalu menebak laju dasar):",
      f"{np.mean([v * (1 - v) for v in prev]):.3f}")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\nTABEL METRIK (gabungan dua arah, anggaran sama per arah; AUC dan Brier dirata-rata dua arah)")
print(tab.round(3).to_string(index=False))
tab.to_csv(OUT_DIR / "metrik_tambahan.csv", index=False)
print("\nCatatan: Brier memakai probabilitas mentah. scale_pos_weight membuat probabilitas XGBoost condong tinggi, jadi Brier tinggi tidak berarti urutan risikonya buruk.")
print("Catatan: Brier untuk gabungan peringkat tidak dihitung karena skornya peringkat, bukan probabilitas.")

# ---------- 2. KURVA PRECISION-RECALL ----------
COL = {"m1": "#4C78A8", "m2b": "#E0662F", "rf": "#59A14F", "ens": "#B07AA1"}
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))
for ax, D, ttl in [(axes[0], D1, "Arah 1: latih data lama, uji data baru"), (axes[1], D2, "Arah 2: latih data baru, uji data lama")]:
    for key in KEYS:
        pr, rc, _ = precision_recall_curve(D["y"], D["prob"][key])
        ax.plot(rc, pr, color=COL[key], lw=1.8, label=f"{NM[key]} (PR-AUC {average_precision_score(D['y'], D['prob'][key]):.2f})")
    tp0 = int(((D["y"] == 1) & (D["p0"] == 1)).sum())
    ax.scatter([tp0 / D["y"].sum()], [tp0 / max(D["p0"].sum(), 1)], color="#222", s=55, marker="D", zorder=5, label="Detektor Pola Wu (satu titik)")
    ax.axhline(D["y"].mean(), color="#999", lw=0.9, ls=":")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(ttl, fontsize=10, loc="left")
    ax.grid(color="#e5e7eb"); ax.legend(fontsize=6.8, loc="upper right")
fig.tight_layout(); fig.savefig(OUT_DIR / "gambar_pr_curve.png", dpi=220, facecolor="white"); plt.close(fig)

# ---------- 3. CONFUSION MATRIX ----------
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.5))
for ax, key in zip(axes, ["wu", "ens"]):
    p = pooled_pred(key); cm = confusion_matrix(y_all, p, labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() * 0.55 else "#1f2937")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(["Normal", "Regression"]); ax.set_yticklabels(["Normal", "Regression"])
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Label sebenarnya"); ax.set_title(NM[key], fontsize=10)
fig.tight_layout(); fig.savefig(OUT_DIR / "gambar_confusion_matrix.png", dpi=220, facecolor="white"); plt.close(fig)

# ---------- 4. PENTINGNYA VARIABEL ----------
Xo = prep(old, ALL_COLS); yo = old[LAB].astype(int).to_numpy()
imp_sum = {}
for s in SEEDS:
    m = xgb(yo, s); m.fit(Xo, yo)
    sc = m.get_booster().get_score(importance_type="gain")
    for k_, v in sc.items(): imp_sum[k_] = imp_sum.get(k_, 0.0) + v / len(SEEDS)
imp = pd.Series(imp_sum).sort_values(ascending=False)
def family(c):
    for pre, nm in [("before_", "Plan sebelum indeks"), ("candidate_", "Plan kandidat"), ("delta_", "Selisih plan"),
                    ("ce_", "CE seluruh plan"), ("loc_", "CE operator berubah")]:
        if c.startswith(pre): return nm
    return "Rasio biaya perkiraan"
fam = imp.groupby([family(c) for c in imp.index]).sum().sort_values(ascending=False)
print("\nPENTINGNYA VARIABEL (gain, rata-rata 5 seed, model dilatih di data lama)")
print("Per kelompok:"); print((fam / fam.sum()).round(3).to_string())
print("15 variabel teratas:"); print(imp.head(15).round(3).to_string())
imp.to_csv(OUT_DIR / "pentingnya_variabel_gain.csv")
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0), gridspec_kw={"width_ratios": [1.25, 1]})
top = imp.head(12)[::-1]
axes[0].barh([c if len(c) < 34 else c[:31] + "..." for c in top.index], top.values, color="#E0662F")
axes[0].set_title("12 variabel teratas (gain)", fontsize=10, loc="left"); axes[0].tick_params(labelsize=8)
fm = (fam / fam.sum())[::-1]
axes[1].barh(fm.index, fm.values * 100, color="#4C78A8"); axes[1].set_title("Porsi gain per kelompok variabel (%)", fontsize=10, loc="left")
axes[1].tick_params(labelsize=8)
for a in axes: a.grid(axis="x", color="#e5e7eb"); a.set_axisbelow(True)
fig.tight_layout(); fig.savefig(OUT_DIR / "gambar_pentingnya_variabel.png", dpi=220, facecolor="white"); plt.close(fig)

# ---------- 5. KURVA KONVERGENSI ----------
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_i, va_i = next(gss.split(Xo, yo, groups=old["template_id"].astype(str)))
mm = xgb(yo[tr_i], 42); mm.fit(Xo.iloc[tr_i], yo[tr_i], eval_set=[(Xo.iloc[tr_i], yo[tr_i]), (Xo.iloc[va_i], yo[va_i])], verbose=False)
ev = mm.evals_result(); tr_ll = ev["validation_0"]["logloss"]; va_ll = ev["validation_1"]["logloss"]
fig, ax = plt.subplots(figsize=(5.6, 3.5))
ax.plot(tr_ll, color="#4C78A8", lw=1.8, label="Latih"); ax.plot(va_ll, color="#E0662F", lw=1.8, label="Validasi (template terpisah)")
ax.set_xlabel("Jumlah pohon"); ax.set_ylabel("Log-loss"); ax.grid(color="#e5e7eb"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "gambar_konvergensi.png", dpi=220, facecolor="white"); plt.close(fig)
print(f"\nKonvergensi: log-loss latih {tr_ll[0]:.3f} -> {tr_ll[-1]:.3f}; validasi {va_ll[0]:.3f} -> {va_ll[-1]:.3f}; "
      f"validasi terendah {min(va_ll):.3f} pada pohon ke-{int(np.argmin(va_ll)) + 1} dari {len(va_ll)}")

with open(OUT_DIR / "ringkasan_metrik.json", "w") as f:
    json.dump({"tabel": tab.round(4).to_dict(orient="records"), "gain_kelompok": (fam / fam.sum()).round(4).to_dict(),
               "gain_top15": imp.head(15).round(4).to_dict()}, f, indent=1, ensure_ascii=False)
print("\nTersimpan di:", OUT_DIR)
print(sorted(p.name for p in OUT_DIR.glob("*")))

## SHAP, analisis kesalahan, dan tuning Optuna


In [ ]:
# ============================================================
# ANALISIS TAMBAHAN = SHAP + ANALISIS KESALAHAN + TUNING OPTUNA (tanpa kebocoran) + TABEL METRIK LENGKAP
# ------------------------------------------------------------
# Satu sel, jalankan setelah figur dan metrik tambahan. Tidak mengubah file lama; hasil masuk ke folder baru:
#   QPR_SF10_Research/artikel_gambar/analisis_lanjutan/
# Isi:
#  1. Tabel metrik lengkap (Acc, Precision, Recall, F1, Macro F1, PR-AUC, ROC-AUC, latensi) untuk
#     Detektor Pola Wu, aturan rasio biaya, regresi logistik, XGBoost x2, random forest, Gabungan peringkat
#  2. SHAP (TreeSHAP) untuk XGBoost Kardinalitas Lokal: ringkasan beeswarm + batang per variabel + per kelompok
#  3. Analisis kesalahan Gabungan vs Detektor Pola Wu (FN dan FP dekat ambang rasio 1,5x)
#  4. Tuning Optuna tanpa kebocoran (validasi silang per template DI DALAM data latih saja),
#     lalu uji pada data sisi lain; dibandingkan dengan pengaturan bawaan
# Label utama: label_v2. Perkiraan waktu: 10 sampai 20 menit di Colab.
# ============================================================
import subprocess, sys
def _pip(*pk):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
try:
    import shap  # noqa
except Exception:
    _pip("shap")
try:
    import optuna  # noqa
except Exception:
    _pip("optuna")

import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import xgboost as xgb_lib
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from scipy.stats import rankdata
warnings.filterwarnings("ignore")

DRIVE_ROOT = Path(os.environ.get("QPR_ROOT", "/content/drive/MyDrive/QPR_SF10_Research"))
OLD_ROOT = DRIVE_ROOT / "incremental"
NEW_ROOT = DRIVE_ROOT / "fresh" / "incremental_fresh"
OUT = DRIVE_ROOT / "artikel_gambar" / "analisis_lanjutan"; OUT.mkdir(parents=True, exist_ok=True)
OLD_FEAT = OLD_ROOT / "incremental_m0_m2b_features.csv"
OLD_V2 = OLD_ROOT / "remeasure" / "remeasure_labels_v2.csv"
NEW_FEAT = NEW_ROOT / "incremental_m0_m2b_features.csv"
NEW_V2 = NEW_ROOT / "remeasure" / "remeasure_labels_v2.csv"
SEEDS = [42, 43, 44, 45, 46]
LAB = "label_v2"
N_TRIALS = int(os.environ.get("QPR_TRIALS", "30"))
for p in [OLD_FEAT, OLD_V2, NEW_FEAT, NEW_V2]:
    assert p.exists(), f"File tidak ditemukan: {p}"

# ---------- data ----------
v2_old = pd.read_csv(OLD_V2); v2_new = pd.read_csv(NEW_V2)
old = pd.read_csv(OLD_FEAT).merge(v2_old[[c for c in v2_old.columns if c in ("sample_id", "label_v1", "label_v2")]],
                                  on="sample_id", how="inner")
new = pd.read_csv(NEW_FEAT)
new["label_v1"] = new["label"].astype(int)
new = new.merge(v2_new[["sample_id", "label_v2"]], on="sample_id", how="left")
old = old[old[LAB].notna()].reset_index(drop=True); new = new[new[LAB].notna()].reset_index(drop=True)
old["batch"] = "lama"; new["batch"] = "baru"

# rasio waktu untuk analisis kesalahan: pakai rasio berulang (v2) kalau ada, kalau tidak rasio satu kali jalan
def pick_ratio(d, v2):
    if "ratio_v2" in v2.columns:                       # rasio median jalan berulang (sesudah/sebelum)
        return d[["sample_id"]].merge(v2[["sample_id", "ratio_v2"]], on="sample_id", how="left")["ratio_v2"].astype(float), "ratio_v2 (file label v2)"
    if "runtime_ratio" in d.columns:
        return d["runtime_ratio"].astype(float), "runtime_ratio (satu kali jalan, lebih berisik)"
    if {"before_elapsed_ms", "after_elapsed_ms"} <= set(d.columns):
        return (d["after_elapsed_ms"] / d["before_elapsed_ms"].clip(lower=1e-9)).astype(float), "after/before elapsed (satu kali jalan)"
    return pd.Series(np.nan, index=d.index), "TIDAK ADA"
old["ratio"], r_old = pick_ratio(old, v2_old); new["ratio"], r_new = pick_ratio(new, v2_new)
print("Sumber rasio waktu -> data lama:", r_old, "| data baru:", r_new)

forbidden = {"before_elapsed_ms", "after_elapsed_ms", "runtime_ratio", "label", "label_v1", "label_v2", "kepastian_v2",
             "label_v1_file", "ratio", "batch"}
def cols_with(prefixes, exact=()):
    return [c for c in old.columns if c not in forbidden and not c.startswith("m0_") and "elapsed_ms" not in c
            and (c.startswith(tuple(prefixes)) or c in exact) and pd.api.types.is_numeric_dtype(old[c])]
BASE = ["before_", "candidate_", "delta_"]
M1_COLS = cols_with(BASE, ("estimated_cost_ratio",))
ALL_COLS = cols_with(BASE + ["ce_", "loc_"], ("estimated_cost_ratio",))
assert all(c in new.columns for c in ALL_COLS)
print(f"Transisi lama {len(old)} (Regression {int(old[LAB].sum())}), baru {len(new)} (Regression {int(new[LAB].sum())}); "
      f"variabel M1 {len(M1_COLS)}, M2b {len(ALL_COLS)}")

def prep(d, cols): return d[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).reset_index(drop=True)
def xgb(y, seed, **extra):
    pos = int((y == 1).sum()); neg = int((y == 0).sum())
    base = dict(objective="binary:logistic", n_estimators=200, learning_rate=0.05, max_depth=3, min_child_weight=1,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, reg_alpha=0.0, scale_pos_weight=neg / max(pos, 1),
                eval_metric="logloss", tree_method="hist", n_jobs=2, random_state=seed, verbosity=0)
    base.update(extra); return XGBClassifier(**base)
def rforest(y, seed):
    return RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=2, class_weight="balanced_subsample",
                                  n_jobs=2, random_state=seed)
def logreg(y, seed):
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000, random_state=seed))
def slog(X): return np.sign(X) * np.log1p(np.abs(X))   # log bertanda untuk variabel berskala besar (khusus regresi logistik)
def rank01(p): return (rankdata(p) - 1) / max(len(p) - 1, 1)
def topk(prob, k):
    p = np.zeros(len(prob), dtype=int); p[np.argsort(-prob, kind="stable")[:k]] = 1; return p
def f1_of(tp, fp, fn): return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

NM = {"wu": "Detektor Pola Wu", "cost": "Aturan rasio biaya perkiraan", "lr": "Regresi logistik",
      "m1": "XGBoost Struktur Plan", "m2b": "XGBoost Kardinalitas Lokal", "rf": "Random forest", "ens": "Gabungan peringkat"}

def direction(tr, te):
    y_tr = tr[LAB].astype(int).to_numpy(); y = te[LAB].astype(int).to_numpy()
    out = {"y": y, "p0": te["m0_prediction"].astype(int).to_numpy(), "prob": {}, "lat_ms": {}, "te": te}
    specs = [("m1", xgb, M1_COLS, False), ("m2b", xgb, ALL_COLS, False), ("rf", rforest, ALL_COLS, False),
             ("lr", logreg, ALL_COLS, True)]
    for key, builder, cols, use_log in specs:
        Xtr, Xte = prep(tr, cols), prep(te, cols)
        if use_log: Xtr, Xte = slog(Xtr), slog(Xte)
        acc = np.zeros(len(te)); t_pred = 0.0
        seeds = SEEDS if key != "lr" else [42]
        for s in seeds:
            m = builder(y_tr, s); m.fit(Xtr, y_tr)
            t0 = time.perf_counter(); acc += m.predict_proba(Xte)[:, 1]; t_pred += time.perf_counter() - t0
        out["prob"][key] = acc / len(seeds); out["lat_ms"][key] = 1000 * (t_pred / len(seeds)) / len(te)
    out["prob"]["cost"] = te["estimated_cost_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(); out["lat_ms"]["cost"] = 0.0
    out["prob"]["ens"] = (rank01(out["prob"]["m1"]) + rank01(out["prob"]["m2b"]) + rank01(out["prob"]["rf"])) / 3.0
    out["lat_ms"]["ens"] = out["lat_ms"]["m1"] + out["lat_ms"]["m2b"] + out["lat_ms"]["rf"]
    out["k"] = int(out["p0"].sum())
    return out

t0 = time.time()
D1 = direction(old, new); D2 = direction(new, old)
print(f"Dua arah selesai ({time.time() - t0:.0f} dtk)")
y_all = np.concatenate([D1["y"], D2["y"]])
KEYS = ["cost", "lr", "m1", "m2b", "rf", "ens"]
def pooled_pred(key, Ds=(D1, D2)):
    if key == "wu": return np.concatenate([D["p0"] for D in Ds])
    return np.concatenate([topk(D["prob"][key], D["k"]) for D in Ds])

# ---------- 1. TABEL METRIK LENGKAP ----------
rows = []
for key in ["wu"] + KEYS:
    p = pooled_pred(key)
    tp = int(((y_all == 1) & (p == 1)).sum()); fp = int(((y_all == 0) & (p == 1)).sum())
    fn = int(((y_all == 1) & (p == 0)).sum()); tn = int(((y_all == 0) & (p == 0)).sum())
    f1p = f1_of(tp, fp, fn); f1n = f1_of(tn, fn, fp)
    r = {"model": NM[key], "TP": tp, "FP": fp, "FN": fn, "TN": tn, "Akurasi": (tp + tn) / len(y_all),
         "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1), "F1_Regression": f1p, "Macro_F1": (f1p + f1n) / 2}
    if key != "wu":
        pa = [average_precision_score(D["y"], D["prob"][key]) for D in (D1, D2)]
        ra = [roc_auc_score(D["y"], D["prob"][key]) for D in (D1, D2)]
        r.update(PR_AUC=float(np.mean(pa)), ROC_AUC=float(np.mean(ra)),
                 Latensi_ms=float(np.mean([D1["lat_ms"][key], D2["lat_ms"][key]])))
    rows.append(r)
tab = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\n=== 1. TABEL METRIK LENGKAP (gabungan dua arah, anggaran tanda sama dengan Detektor Pola Wu) ===")
print(tab.round(3).to_string(index=False))
tab.to_csv(OUT / "tabel_metrik_lengkap.csv", index=False)

# ---------- 2. SHAP ----------
def pretty(c):
    for pre, nm in [("before_", "Plan sebelum: "), ("candidate_", "Plan kandidat: "), ("delta_", "Selisih: "),
                    ("ce_", "CE plan: "), ("loc_", "CE berubah: ")]:
        if c.startswith(pre): return nm + c[len(pre):].replace("_", " ")
    return "Rasio biaya perkiraan" if c == "estimated_cost_ratio" else c.replace("_", " ")
def family(c):
    for pre, nm in [("before_", "Plan sebelum indeks"), ("candidate_", "Plan kandidat"), ("delta_", "Selisih plan"),
                    ("ce_", "CE seluruh plan"), ("loc_", "CE operator berubah")]:
        if c.startswith(pre): return nm
    return "Rasio biaya perkiraan"

print("\n=== 2. SHAP (TreeSHAP, XGBoost Kardinalitas Lokal) ===")
try:
    sv_list, X_list = [], []
    for tr, te in [(old, new), (new, old)]:
        y_tr = tr[LAB].astype(int).to_numpy(); Xtr, Xte = prep(tr, ALL_COLS), prep(te, ALL_COLS)
        acc = np.zeros((len(te), len(ALL_COLS)))
        for s in SEEDS:
            m = xgb(y_tr, s); m.fit(Xtr, y_tr)
            contrib = m.get_booster().predict(xgb_lib.DMatrix(Xte, feature_names=list(ALL_COLS)), pred_contribs=True)
            acc += contrib[:, :-1] / len(SEEDS)           # kolom terakhir = bias
        sv_list.append(acc); X_list.append(Xte)
    SV = np.vstack(sv_list); XX = pd.concat(X_list, ignore_index=True)
    mean_abs = pd.Series(np.abs(SV).mean(axis=0), index=ALL_COLS).sort_values(ascending=False)
    fam_abs = mean_abs.groupby([family(c) for c in mean_abs.index]).sum().sort_values(ascending=False)
    print("Rata-rata |SHAP| per kelompok (porsi):"); print((fam_abs / fam_abs.sum()).round(3).to_string())
    print("15 variabel teratas (rata-rata |SHAP|, satuan log-odds):"); print(mean_abs.head(15).round(4).to_string())
    pd.DataFrame({"variabel": mean_abs.index, "mean_abs_shap": mean_abs.values, "kelompok": [family(c) for c in mean_abs.index]}
                 ).to_csv(OUT / "shap_rata_abs.csv", index=False)
    # arah efek: korelasi nilai variabel dengan SHAP-nya (tanda sederhana)
    tops = list(mean_abs.head(15).index)
    arah = {}
    for c in tops:
        j = ALL_COLS.index(c)
        cc = np.corrcoef(rank01(XX[c].to_numpy()), SV[:, j])[0, 1] if XX[c].nunique() > 1 else np.nan
        arah[c] = round(float(cc), 2)
    print("Korelasi nilai variabel dengan SHAP (positif = nilai makin besar, risiko makin naik):"); print(pd.Series(arah).to_string())
    # beeswarm
    try:
        import shap
        names = [pretty(c) for c in ALL_COLS]
        plt.figure(figsize=(7.6, 5.4))
        shap.summary_plot(SV, XX.values, feature_names=names, max_display=15, show=False, plot_size=None)
        plt.xlabel("Nilai SHAP (dampak pada log-odds Regression)", fontsize=9)
        plt.tight_layout(); plt.savefig(OUT / "gambar_shap_summary.png", dpi=220, facecolor="white"); plt.close()
    except Exception as e:
        print("shap.summary_plot gagal, memakai beeswarm sederhana:", repr(e)[:120])
        fig, ax = plt.subplots(figsize=(7.6, 5.4)); rng = np.random.RandomState(0)
        for i, c in enumerate(tops[::-1]):
            j = ALL_COLS.index(c); r = rank01(XX[c].to_numpy())
            ax.scatter(SV[:, j], i + rng.uniform(-0.25, 0.25, len(r)), c=r, cmap="coolwarm", s=7)
        ax.set_yticks(range(len(tops))); ax.set_yticklabels([pretty(c) for c in tops[::-1]], fontsize=8)
        ax.set_xlabel("Nilai SHAP (dampak pada log-odds Regression)"); ax.axvline(0, color="#999", lw=0.8)
        fig.tight_layout(); fig.savefig(OUT / "gambar_shap_summary.png", dpi=220, facecolor="white"); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0), gridspec_kw={"width_ratios": [1.25, 1]})
    top = mean_abs.head(12)[::-1]
    axes[0].barh([pretty(c) for c in top.index], top.values, color="#E0662F")
    axes[0].set_title("12 variabel teratas (rata-rata |SHAP|)", fontsize=10, loc="left"); axes[0].tick_params(labelsize=8)
    fm = (fam_abs / fam_abs.sum())[::-1]
    axes[1].barh(fm.index, fm.values * 100, color="#4C78A8"); axes[1].set_title("Porsi |SHAP| per kelompok variabel (%)", fontsize=10, loc="left")
    axes[1].tick_params(labelsize=8)
    for a in axes: a.grid(axis="x", color="#e5e7eb"); a.set_axisbelow(True)
    fig.tight_layout(); fig.savefig(OUT / "gambar_shap_batang.png", dpi=220, facecolor="white"); plt.close(fig)
    shap_ok = True
except Exception as e:
    shap_ok = False; print("SHAP GAGAL:", repr(e)[:300])

# ---------- 3. ANALISIS KESALAHAN ----------
print("\n=== 3. ANALISIS KESALAHAN (Gabungan peringkat vs Detektor Pola Wu, dua arah digabung, anggaran sama) ===")
parts = []
for dname, D in [("lama->baru", D1), ("baru->lama", D2)]:
    te = D["te"]; flag_ens = topk(D["prob"]["ens"], D["k"])
    parts.append(pd.DataFrame({"sample_id": te["sample_id"].to_numpy(), "template_id": te["template_id"].astype(str).to_numpy(),
                               "arah": dname, "label": D["y"], "rasio_waktu": te["ratio"].to_numpy(),
                               "wu_tandai": D["p0"], "ens_tandai": flag_ens,
                               "ens_skor": D["prob"]["ens"], "ens_peringkat": (-D["prob"]["ens"]).argsort().argsort() + 1,
                               "rasio_biaya_perkiraan": te["estimated_cost_ratio"].to_numpy()}))
E = pd.concat(parts, ignore_index=True)
E.to_csv(OUT / "analisis_kesalahan_per_transisi.csv", index=False)
reg = E[E.label == 1]; nor = E[E.label == 0]
print("Tumpang tindih pada transisi Regression nyata (n=%d):" % len(reg))
print(f"  ditemukan keduanya {int(((reg.wu_tandai==1)&(reg.ens_tandai==1)).sum())}, hanya Wu {int(((reg.wu_tandai==1)&(reg.ens_tandai==0)).sum())}, "
      f"hanya Gabungan {int(((reg.wu_tandai==0)&(reg.ens_tandai==1)).sum())}, tidak ada yang menemukan {int(((reg.wu_tandai==0)&(reg.ens_tandai==0)).sum())}")
print("Tumpang tindih pada alarm palsu: Wu %d, Gabungan %d, keduanya %d" % (
    int((nor.wu_tandai==1).sum()), int((nor.ens_tandai==1).sum()), int(((nor.wu_tandai==1)&(nor.ens_tandai==1)).sum())))
have_ratio = E.rasio_waktu.notna().any()
if have_ratio:
    bins = [0, 1.0, 1.2, 1.5, 2, 3, 5, np.inf]; lab = ["<1,0", "1,0-1,2", "1,2-1,5", "1,5-2", "2-3", "3-5", ">=5"]
    E["bin_rasio"] = pd.cut(E.rasio_waktu, bins=bins, labels=lab, right=False)
    g = E.groupby("bin_rasio", observed=True).agg(n=("label", "size"), regression=("label", "sum"),
                                                  wu_tandai=("wu_tandai", "sum"), gabungan_tandai=("ens_tandai", "sum"))
    print("\nTransisi per pita rasio waktu (rasio = sesudah/sebelum; Regression = rasio >= 1,5):"); print(g.to_string())
    print("Catatan: pita 1,2-1,5 dan 1,5-2 adalah daerah batas; label berasal dari median jalan berulang, rasio di sini bisa berbeda tipis.")
    g.to_csv(OUT / "analisis_kesalahan_pita_rasio.csv")
tmpl = E.groupby("template_id").apply(lambda d: pd.Series({
    "n": len(d), "regression": int(d.label.sum()),
    "TP_wu": int(((d.label==1)&(d.wu_tandai==1)).sum()), "TP_gab": int(((d.label==1)&(d.ens_tandai==1)).sum()),
    "FP_wu": int(((d.label==0)&(d.wu_tandai==1)).sum()), "FP_gab": int(((d.label==0)&(d.ens_tandai==1)).sum())}))
tmpl = tmpl.sort_values("regression", ascending=False)
print("\nPer template (10 teratas menurut jumlah Regression):"); print(tmpl.head(10).to_string())
tmpl.to_csv(OUT / "analisis_kesalahan_per_template.csv")
fn = reg[reg.ens_tandai == 0].sort_values("ens_peringkat", ascending=False)
fp = nor[nor.ens_tandai == 1].sort_values("ens_peringkat")
cols_show = ["template_id", "arah", "rasio_waktu", "wu_tandai", "ens_peringkat", "rasio_biaya_perkiraan"]
print("\nFalse Negative Gabungan (Regression yang lolos), 10 paling jauh dari daftar tandai:"); print(fn[cols_show].head(10).round(2).to_string(index=False))
print("False Positive Gabungan (alarm palsu), 10 skor tertinggi:"); print(fp[cols_show].head(10).round(2).to_string(index=False))
if have_ratio:
    fpr = fp.rasio_waktu.dropna(); fnr = fn.rasio_waktu.dropna()
    if len(fpr): print(f"Alarm palsu Gabungan: {len(fpr)}; rasio >=1,2 (dekat ambang) {int((fpr>=1.2).sum())}; rasio <1,0 (benar-benar tidak melambat) {int((fpr<1.0).sum())}")
    if len(fnr): print(f"Regression lolos Gabungan: {len(fnr)}; rasio 1,5-2 (dekat ambang) {int(((fnr>=1.5)&(fnr<2)).sum())}; rasio >=3 {int((fnr>=3).sum())}")

# ---------- 4. OPTUNA TANPA KEBOCORAN ----------
print(f"\n=== 4. TUNING OPTUNA ({N_TRIALS} percobaan, validasi silang per template di dalam data latih saja) ===")
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
def cv_score(tr, params):
    y = tr[LAB].astype(int).to_numpy(); X = prep(tr, ALL_COLS); g = tr["template_id"].astype(str).to_numpy()
    sc = []
    for a, b in StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42).split(X, y, g):
        if y[b].sum() == 0 or y[a].sum() == 0: continue
        m = xgb(y[a], 42, **params); m.fit(X.iloc[a], y[a])
        sc.append(average_precision_score(y[b], m.predict_proba(X.iloc[b])[:, 1]))
    return float(np.mean(sc)) if sc else 0.0
def tune(tr):
    def obj(trial):
        pr = dict(n_estimators=trial.suggest_int("n_estimators", 100, 400, step=50),
                  learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                  max_depth=trial.suggest_int("max_depth", 2, 6),
                  min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                  subsample=trial.suggest_float("subsample", 0.5, 1.0),
                  colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                  reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
                  gamma=trial.suggest_float("gamma", 0.0, 5.0))
        return cv_score(tr, pr)
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    st.optimize(obj, n_trials=N_TRIALS, show_progress_bar=False)
    return st
res = []; best_all = {}; hist_all = {}
for dname, tr, te, D in [("lama->baru", old, new, D1), ("baru->lama", new, old, D2)]:
    t0 = time.time(); st = tune(tr); bp = st.best_params
    base_cv = cv_score(tr, {})
    y_tr = tr[LAB].astype(int).to_numpy(); Xtr, Xte = prep(tr, ALL_COLS), prep(te, ALL_COLS); y = te[LAB].astype(int).to_numpy()
    pt = np.zeros(len(te)); t_pred = 0.0
    for s in SEEDS:
        m = xgb(y_tr, s, **bp); m.fit(Xtr, y_tr)
        t1 = time.perf_counter(); pt += m.predict_proba(Xte)[:, 1] / len(SEEDS); t_pred += time.perf_counter() - t1
    best_all[dname] = {"params": bp, "cv_pr_auc_tuned": st.best_value, "cv_pr_auc_default": base_cv}
    hist_all[dname] = [t.value for t in st.trials]
    D["prob"]["m2b_tuned"] = pt
    print(f"[{dname}] PR-AUC validasi silang: bawaan {base_cv:.3f} -> terbaik {st.best_value:.3f} ({time.time()-t0:.0f} dtk); parameter: "
          + ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in bp.items()))
cmp_rows = []
for key, nm in [("m2b", "XGBoost Kardinalitas Lokal (bawaan)"), ("m2b_tuned", "XGBoost Kardinalitas Lokal (Optuna)")]:
    p = np.concatenate([topk(D["prob"][key], D["k"]) for D in (D1, D2)])
    tp = int(((y_all == 1) & (p == 1)).sum()); fp_ = int(((y_all == 0) & (p == 1)).sum()); fn_ = int(((y_all == 1) & (p == 0)).sum())
    tn = int(((y_all == 0) & (p == 0)).sum()); f1p = f1_of(tp, fp_, fn_); f1n = f1_of(tn, fn_, fp_)
    cmp_rows.append({"model": nm, "TP": tp, "FP": fp_, "FN": fn_, "TN": tn, "Precision": tp / max(tp + fp_, 1), "Recall": tp / max(tp + fn_, 1),
                     "F1": f1p, "Macro_F1": (f1p + f1n) / 2,
                     "PR_AUC_arah1": average_precision_score(D1["y"], D1["prob"][key]),
                     "PR_AUC_arah2": average_precision_score(D2["y"], D2["prob"][key]),
                     "ROC_AUC_arah1": roc_auc_score(D1["y"], D1["prob"][key]), "ROC_AUC_arah2": roc_auc_score(D2["y"], D2["prob"][key])})
cmp_ = pd.DataFrame(cmp_rows); print("\nPerbandingan di data uji sisi lain (tanpa kebocoran):"); print(cmp_.round(3).to_string(index=False))
cmp_.to_csv(OUT / "perbandingan_optuna.csv", index=False)
fig, ax = plt.subplots(figsize=(5.6, 3.5))
for dname, c in zip(hist_all, ["#4C78A8", "#E0662F"]):
    h = np.array(hist_all[dname]); ax.plot(np.arange(1, len(h) + 1), np.maximum.accumulate(h), color=c, lw=1.8, label=f"Arah {dname}")
ax.set_xlabel("Percobaan Optuna"); ax.set_ylabel("PR-AUC validasi silang terbaik"); ax.grid(color="#e5e7eb"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "gambar_optuna_riwayat.png", dpi=220, facecolor="white"); plt.close(fig)

with open(OUT / "ringkasan_analisis.json", "w") as f:
    json.dump({"tabel": tab.round(4).to_dict(orient="records"), "optuna": best_all, "perbandingan_optuna": cmp_.round(4).to_dict(orient="records"),
               "shap_top15": (mean_abs.head(15).round(4).to_dict() if shap_ok else None),
               "shap_kelompok": ((fam_abs / fam_abs.sum()).round(4).to_dict() if shap_ok else None)}, f, indent=1, ensure_ascii=False, default=str)
print("\nTersimpan di:", OUT); print(sorted(p.name for p in OUT.glob("*")))

## Analisis kesalahan menurut rasio waktu


In [ ]:
# ============================================================
# ANALISIS KESALAHAN BERDASARKAN RASIO WAKTU (ratio_v2)
# Rasio waktu diambil dari kolom "ratio_v2" (median jalan berulang), bukan dari kolom "iteration".
# Membaca hasil analisis kesalahan sebelumnya + file label v2 dari Drive, lalu menghitung ulang tabel pita rasio dan satu gambar.
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/QPR_SF10_Research"); OUT = ROOT / "artikel_gambar" / "analisis_lanjutan"
E = pd.read_csv(OUT / "analisis_kesalahan_per_transisi.csv")
lab_old = pd.read_csv(ROOT / "incremental" / "remeasure" / "remeasure_labels_v2.csv")
lab_new = pd.read_csv(ROOT / "fresh" / "incremental_fresh" / "remeasure" / "remeasure_labels_v2.csv")
print("kolom file label:", list(lab_old.columns))
assert "ratio_v2" in lab_old.columns and "ratio_v2" in lab_new.columns, "kolom ratio_v2 tidak ada; periksa daftar kolom di atas"
parts = []
for arah, lab in [("lama->baru", lab_new), ("baru->lama", lab_old)]:      # batch uji tiap arah
    e = E[E["arah"] == arah].drop(columns=["rasio_waktu"], errors="ignore")
    parts.append(e.merge(lab[["sample_id", "ratio_v2"]], on="sample_id", how="left"))
E = pd.concat(parts, ignore_index=True).rename(columns={"ratio_v2": "rasio_v2"})
print("baris:", len(E), "| rasio kosong:", int(E.rasio_v2.isna().sum()))
chk = E[E.label == 1].rasio_v2.min(), E[E.label == 0].rasio_v2.max()
print(f"cek label: rasio terkecil di kelas Regression = {chk[0]:.2f} (harus >= 1,5); rasio terbesar di kelas Normal = {chk[1]:.2f} (harus < 1,5)")
bins = [0, 0.8, 1.0, 1.2, 1.5, 2, 3, 5, np.inf]; names = ["<0,8", "0,8-1,0", "1,0-1,2", "1,2-1,5", "1,5-2", "2-3", "3-5", ">=5"]
E["pita"] = pd.cut(E.rasio_v2, bins=bins, labels=names, right=False)
g = E.groupby("pita", observed=False).agg(n=("label", "size"), regression=("label", "sum"), wu_tandai=("wu_tandai", "sum"),
                                          gabungan_tandai=("ens_tandai", "sum"))
print("\nPer pita rasio waktu sebenarnya (sesudah/sebelum, median berulang):"); print(g.to_string())
r = E[E.label == 1]; n_ = E[E.label == 0]
tab = []
for nm, lo, hi in [("1,5-2", 1.5, 2), ("2-3", 2, 3), ("3-5", 3, 5), (">=5", 5, np.inf)]:
    s = r[(r.rasio_v2 >= lo) & (r.rasio_v2 < hi)]
    tab.append({"pita": nm, "jumlah_regression": len(s), "tertangkap_wu": int(s.wu_tandai.sum()), "tertangkap_gabungan": int(s.ens_tandai.sum())})
tab = pd.DataFrame(tab); print("\nRecall menurut tingkat keparahan (hanya transisi Regression):"); print(tab.to_string(index=False))
fa = n_[n_.ens_tandai == 1]; fw = n_[n_.wu_tandai == 1]
def band(x):
    return pd.Series({"total": len(x), "<1,0": int((x.rasio_v2 < 1.0).sum()), "1,0-1,2": int(((x.rasio_v2 >= 1.0) & (x.rasio_v2 < 1.2)).sum()),
                      "1,2-1,5 (dekat ambang)": int(((x.rasio_v2 >= 1.2) & (x.rasio_v2 < 1.5)).sum())})
fp_tab = pd.DataFrame({"Detektor Pola Wu": band(fw), "Gabungan peringkat": band(fa)}).T
print("\nAlarm palsu menurut rasio waktu sebenarnya:"); print(fp_tab.to_string())
fn = r[r.ens_tandai == 0]
print(f"\nRegression yang lolos Gabungan: {len(fn)}; rasio 1,5-2 (dekat ambang) {int(((fn.rasio_v2>=1.5)&(fn.rasio_v2<2)).sum())}, "
      f"2-3 {int(((fn.rasio_v2>=2)&(fn.rasio_v2<3)).sum())}, >=3 {int((fn.rasio_v2>=3).sum())}; median rasio {fn.rasio_v2.median():.2f} "
      f"(Regression yang tertangkap Gabungan: {r[r.ens_tandai==1].rasio_v2.median():.2f}; yang tertangkap Wu: {r[r.wu_tandai==1].rasio_v2.median():.2f})")
print("Regression terparah yang lolos Gabungan (10 teratas):")
print(fn.sort_values("rasio_v2", ascending=False)[["template_id", "arah", "rasio_v2", "wu_tandai", "ens_peringkat", "rasio_biaya_perkiraan"]].head(10).round(2).to_string(index=False))
print("Alarm palsu Gabungan dengan rasio tertinggi (10 teratas):")
print(fa.sort_values("rasio_v2", ascending=False)[["template_id", "arah", "rasio_v2", "wu_tandai", "ens_peringkat", "rasio_biaya_perkiraan"]].head(10).round(2).to_string(index=False))
E.to_csv(OUT / "analisis_kesalahan_per_transisi_rasio_waktu.csv", index=False); g.to_csv(OUT / "analisis_kesalahan_pita_rasio_waktu.csv")
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.6))
x = np.arange(len(tab)); w = 0.38
ax[0].bar(x - w/2, tab.tertangkap_wu, w, color="#6b7280", label="Detektor Pola Wu"); ax[0].bar(x + w/2, tab.tertangkap_gabungan, w, color="#B07AA1", label="Gabungan peringkat")
for i, t in enumerate(tab.jumlah_regression): ax[0].text(i, max(tab.tertangkap_wu[i], tab.tertangkap_gabungan[i]) + 0.3, f"dari {t}", ha="center", fontsize=8)
ax[0].set_xticks(x); ax[0].set_xticklabels(tab.pita); ax[0].set_xlabel("Rasio waktu sebenarnya (sesudah/sebelum)"); ax[0].set_ylabel("Regression tertangkap")
ax[0].set_title("Regression tertangkap menurut keparahan", fontsize=10, loc="left"); ax[0].legend(fontsize=8); ax[0].grid(axis="y", color="#e5e7eb"); ax[0].set_axisbelow(True)
cols = ["<1,0", "1,0-1,2", "1,2-1,5 (dekat ambang)"]; x2 = np.arange(len(cols))
ax[1].bar(x2 - w/2, fp_tab.loc["Detektor Pola Wu", cols], w, color="#6b7280", label="Detektor Pola Wu"); ax[1].bar(x2 + w/2, fp_tab.loc["Gabungan peringkat", cols], w, color="#B07AA1", label="Gabungan peringkat")
ax[1].set_xticks(x2); ax[1].set_xticklabels(cols, fontsize=8); ax[1].set_xlabel("Rasio waktu sebenarnya"); ax[1].set_ylabel("Alarm palsu")
ax[1].set_title("Alarm palsu menurut rasio waktu sebenarnya", fontsize=10, loc="left"); ax[1].grid(axis="y", color="#e5e7eb"); ax[1].set_axisbelow(True)
fig.tight_layout(); fig.savefig(OUT / "gambar_galat_keparahan.png", dpi=220, facecolor="white"); plt.close(fig)